<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/16_Privacy_%26_Disclosure_Risk_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# =====================================================================
# NOTEBOOK 16 — SECTION 1: HEADER, SCOPE & CONFIGURATION (FROZEN V4)
# =====================================================================

from pathlib import Path
from datetime import datetime, timezone
import os
import json
import hashlib
import warnings
import gc
import shutil

import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

# ---------------------------------------------------------------------
# 1. CANONICAL PROJECT ROOT
# ---------------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Research")

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Project root not found: {PROJECT_ROOT}")

NOTEBOOK_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_16"

DIRS = {
    "configuration": NOTEBOOK_ROOT / "configuration",
    "inputs": NOTEBOOK_ROOT / "inputs",
    "exact": NOTEBOOK_ROOT / "exact_duplicate_analysis",
    "near": NOTEBOOK_ROOT / "near_duplicate_analysis",
    "distance": NOTEBOOK_ROOT / "nearest_neighbor_analysis",
    "disclosure": NOTEBOOK_ROOT / "disclosure_risk",
    "attacks": NOTEBOOK_ROOT / "privacy_attacks",
    "summary": NOTEBOOK_ROOT / "summaries",
    "results": NOTEBOOK_ROOT / "results",
    "validation": NOTEBOOK_ROOT / "validation",
    "logs": NOTEBOOK_ROOT / "logs",
}

for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# 2. DATASET REGISTRY AND EVALUATION SETTINGS
# ---------------------------------------------------------------------

DATASETS = {
    "adult_income": {"target": "income"},
    "bank_marketing": {"target": "y"},
    "diabetes_130us": {"target": "readmitted"},
}

MASTER_SEED = 2025
PRIVACY_SEED = 1616
MAX_REAL_REFERENCE_ROWS = 5000
MAX_SYNTHETIC_ROWS = 5000
PAIRWISE_CHUNK_SIZE = 128
NEAR_DUPLICATE_THRESHOLD = 0.95
RANDOM_SAMPLE_FOR_EXPENSIVE_ANALYSES = True
SYNTHETIC_GENERATION_REPETITIONS = 1

RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat()

# ---------------------------------------------------------------------
# 3. DISCOVER EXISTING PROJECT CONFIGURATION AND NOTEBOOK ARTIFACTS
# ---------------------------------------------------------------------

CONFIG_FILENAMES = {"master_config.yaml", "master_config.yml"}
SEARCH_EXCLUDED = {
    ".ipynb_checkpoints",
    "site-packages",
    "node_modules",
}

CONFIG_CANDIDATES = []
for root, dirs, files in os.walk(PROJECT_ROOT):
    dirs[:] = [d for d in dirs if d not in SEARCH_EXCLUDED]
    for filename in files:
        if filename.lower() in CONFIG_FILENAMES:
            candidate = Path(root) / filename
            if "notebook_16" not in str(candidate).lower():
                CONFIG_CANDIDATES.append(candidate)

CONFIG_CANDIDATES = sorted(
    set(CONFIG_CANDIDATES),
    key=lambda p: (len(p.parts), str(p)),
)

CONFIG_PATH = CONFIG_CANDIDATES[0] if CONFIG_CANDIDATES else None
MASTER_CONFIG = {}
CONFIG_STATUS = "NOT_FOUND"

if CONFIG_PATH is not None:
    import yaml

    with open(CONFIG_PATH, "r", encoding="utf-8") as file:
        loaded_config = yaml.safe_load(file)

    if not isinstance(loaded_config, dict):
        raise ValueError(f"Invalid YAML configuration: {CONFIG_PATH}")

    MASTER_CONFIG = loaded_config
    CONFIG_STATUS = "LOADED"

# Search for persisted Notebook 00 configuration/artifacts.
NOTEBOOK_00_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_00"

NOTEBOOK_00_CONFIG_CANDIDATES = []
for root, dirs, files in os.walk(PROJECT_ROOT):
    dirs[:] = [d for d in dirs if d not in SEARCH_EXCLUDED]
    for filename in files:
        low = filename.lower()
        if (
            "notebook_00" in str(root).lower()
            and low.endswith((".yaml", ".yml", ".json"))
            and "config" in low
        ):
            NOTEBOOK_00_CONFIG_CANDIDATES.append(Path(root) / filename)

# Do not treat Notebook 16's own config as a master configuration.
NOTEBOOK_00_CONFIG_CANDIDATES = sorted(
    set(NOTEBOOK_00_CONFIG_CANDIDATES),
    key=lambda p: (len(p.parts), str(p)),
)

# ---------------------------------------------------------------------
# 4. SAVE FROZEN NOTEBOOK CONFIGURATION
# ---------------------------------------------------------------------

NOTEBOOK_CONFIG = {
    "notebook_id": "notebook_16",
    "title": "Privacy & Disclosure Risk Evaluation",
    "project_root": str(PROJECT_ROOT),
    "notebook_root": str(NOTEBOOK_ROOT),
    "master_config_path": str(CONFIG_PATH) if CONFIG_PATH else None,
    "master_config_status": CONFIG_STATUS,
    "master_config_candidates": [str(p) for p in CONFIG_CANDIDATES],
    "notebook_00_config_candidates": [
        str(p) for p in NOTEBOOK_00_CONFIG_CANDIDATES
    ],
    "datasets": DATASETS,
    "master_seed": MASTER_SEED,
    "privacy_seed": PRIVACY_SEED,
    "max_real_reference_rows": MAX_REAL_REFERENCE_ROWS,
    "max_synthetic_rows": MAX_SYNTHETIC_ROWS,
    "pairwise_chunk_size": PAIRWISE_CHUNK_SIZE,
    "near_duplicate_threshold": NEAR_DUPLICATE_THRESHOLD,
    "synthetic_generation_repetitions": SYNTHETIC_GENERATION_REPETITIONS,
    "privacy_attack_policy": (
        "Run only when an explicit, documented attack protocol is configured."
    ),
    "dp_claim_policy": (
        "Empirical disclosure-risk indicators do not establish formal "
        "differential-privacy guarantees."
    ),
    "run_started_utc": RUN_STARTED_UTC,
}

CONFIG_OUTPUT_PATH = DIRS["configuration"] / "notebook_16_config.json"

with open(CONFIG_OUTPUT_PATH, "w", encoding="utf-8") as file:
    json.dump(NOTEBOOK_CONFIG, file, indent=2, allow_nan=False)

# Preserve the authoritative master config if found.
MASTER_CONFIG_SNAPSHOT_PATH = None
if CONFIG_PATH is not None:
    MASTER_CONFIG_SNAPSHOT_PATH = (
        DIRS["configuration"] / "master_config_snapshot.yaml"
    )
    shutil.copy2(CONFIG_PATH, MASTER_CONFIG_SNAPSHOT_PATH)

# ---------------------------------------------------------------------
# 5. SECTION 1 VALIDATION
# ---------------------------------------------------------------------

SECTION_1_CHECKS = [
    {
        "check": "project_root_exists",
        "status": "PASS" if PROJECT_ROOT.is_dir() else "FAIL",
        "observed": str(PROJECT_ROOT),
    },
    {
        "check": "dataset_registry",
        "status": (
            "PASS"
            if set(DATASETS) == {
                "adult_income", "bank_marketing", "diabetes_130us"
            }
            else "FAIL"
        ),
        "observed": json.dumps(DATASETS),
    },
    {
        "check": "master_configuration",
        "status": "PASS" if CONFIG_STATUS == "LOADED" else "WARNING",
        "observed": str(CONFIG_PATH) if CONFIG_PATH else "Not found",
    },
    {
        "check": "notebook_00_configuration_candidates",
        "status": (
            "PASS" if NOTEBOOK_00_CONFIG_CANDIDATES else "WARNING"
        ),
        "observed": json.dumps(
            [str(p) for p in NOTEBOOK_00_CONFIG_CANDIDATES]
        ),
    },
    {
        "check": "notebook_config_saved",
        "status": (
            "PASS"
            if CONFIG_OUTPUT_PATH.is_file()
            and CONFIG_OUTPUT_PATH.stat().st_size > 0
            else "FAIL"
        ),
        "observed": str(CONFIG_OUTPUT_PATH),
    },
]

SECTION_1_VALIDATION_DF = pd.DataFrame(SECTION_1_CHECKS)
SECTION_1_VALIDATION_PATH = DIRS["validation"] / "section_01_validation.csv"
SECTION_1_VALIDATION_DF.to_csv(SECTION_1_VALIDATION_PATH, index=False)

# Reload persisted outputs to verify integrity.
with open(CONFIG_OUTPUT_PATH, "r", encoding="utf-8") as file:
    CONFIG_RELOADED = json.load(file)

VALIDATION_RELOADED = pd.read_csv(SECTION_1_VALIDATION_PATH)

CONFIG_SAVE_PASS = (
    CONFIG_RELOADED.get("notebook_id") == "notebook_16"
    and CONFIG_RELOADED.get("datasets") == DATASETS
    and CONFIG_RELOADED.get("master_config_status") == CONFIG_STATUS
)

CORE_CHECKS = [
    "project_root_exists",
    "dataset_registry",
    "notebook_config_saved",
]

CORE_CHECKS_PASS = bool(
    SECTION_1_VALIDATION_DF.loc[
        SECTION_1_VALIDATION_DF["check"].isin(CORE_CHECKS), "status"
    ].eq("PASS").all()
)

assert CORE_CHECKS_PASS, (
    f"Section 1 core validation failed: {SECTION_1_VALIDATION_PATH}"
)
assert CONFIG_SAVE_PASS, "Configuration reload verification failed."

# ---------------------------------------------------------------------
# 6. DISPLAY STATUS
# ---------------------------------------------------------------------

print("=" * 80)
print("NOTEBOOK 16 — SECTION 1 VALIDATION")
print("=" * 80)
print("Project root:", PROJECT_ROOT)
print("Notebook root:", NOTEBOOK_ROOT)
print("Master config status:", CONFIG_STATUS)
print("Master config path:", CONFIG_PATH)
print("Notebook 00 config candidates:", len(NOTEBOOK_00_CONFIG_CANDIDATES))
print("Configuration saved:", CONFIG_OUTPUT_PATH)
print("Validation saved:", SECTION_1_VALIDATION_PATH)
print("Config reload verification:", "PASS" if CONFIG_SAVE_PASS else "FAIL")
print("Core validation:", "PASS" if CORE_CHECKS_PASS else "FAIL")
print("=" * 80)

display(VALIDATION_RELOADED)

if CONFIG_STATUS != "LOADED" and not NOTEBOOK_00_CONFIG_CANDIDATES:
    print(
        "BLOCKER: Neither master_config.yaml nor a recognizable Notebook 00 "
        "configuration artifact was found. Section 1 initialization is "
        "persisted, but confirm the authoritative configuration and project "
        "artifacts before proceeding with privacy evaluation."
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NOTEBOOK 16 — SECTION 1 VALIDATION
Project root: /content/drive/MyDrive/SPP_GAN_Research
Notebook root: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16
Master config status: NOT_FOUND
Master config path: None
Notebook 00 config candidates: 5
Configuration saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/configuration/notebook_16_config.json
Validation saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/validation/section_01_validation.csv
Config reload verification: PASS
Core validation: PASS


,check,status,observed
0,project_root_exists,PASS,/content/drive/MyDrive/SPP_GAN_Research
1,dataset_registry,PASS,"{""adult_income"": {""target"": ""income""}, ""bank_m..."
2,master_configuration,WARNING,Not found
3,notebook_00_configuration_candidates,PASS,"[""/content/drive/MyDrive/SPP_GAN_Research/resu..."
4,notebook_config_saved,PASS,/content/drive/MyDrive/SPP_GAN_Research/result...


In [4]:
# =====================================================================
# NOTEBOOK 16 — SECTION 2: LOAD REAL DATA (FROZEN V2)
# =====================================================================

import hashlib
import json
from pathlib import Path

# ---------------------------------------------------------------------
# 1. AUTHORITATIVE RAW-DATA PATHS
# ---------------------------------------------------------------------

REAL_DATA_PATH_CANDIDATES = {
    "adult_income": [
        PROJECT_ROOT / "data" / "raw" / "adult_income.csv",
        PROJECT_ROOT / "data" / "processed" / "adult_income_processed.csv",
    ],
    "bank_marketing": [
        PROJECT_ROOT / "data" / "raw" / "bank_marketing.csv",
        PROJECT_ROOT / "data" / "processed" / "bank_marketing_processed.csv",
    ],
    "diabetes_130us": [
        PROJECT_ROOT / "data" / "raw" / "diabetes_130us.csv",
        PROJECT_ROOT / "data" / "processed" / "diabetes_130us_processed.csv",
    ],
}

EXPECTED_MIN_COLUMNS = {
    "adult_income": 15,
    "bank_marketing": 17,
    "diabetes_130us": 50,
}

# ---------------------------------------------------------------------
# 2. HELPERS
# ---------------------------------------------------------------------

def section2_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def section2_is_synthetic(path):
    parts = [part.lower() for part in path.parts]
    name = path.name.lower()
    return (
        "synthetic" in parts
        or "synthetic" in name
        or "generated" in name
    )


def section2_read_csv(path, dataset_name):
    # Bank Marketing source files may use semicolon delimiters.
    # Detect delimiter from a small sample rather than loading twice.
    with open(path, "r", encoding="utf-8-sig", errors="replace") as file:
        sample = file.read(8192)

    delimiter = ";" if sample.count(";") > sample.count(",") else ","

    frame = pd.read_csv(
        path,
        sep=delimiter,
        low_memory=False,
        encoding="utf-8-sig",
    )

    if frame.empty:
        raise ValueError(f"{dataset_name}: source file is empty: {path}")

    if frame.columns.duplicated().any():
        duplicate_columns = frame.columns[
            frame.columns.duplicated()
        ].tolist()
        raise ValueError(
            f"{dataset_name}: duplicate column names: {duplicate_columns}"
        )

    return frame


# ---------------------------------------------------------------------
# 3. LOAD REAL DATA AND VALIDATE SOURCE STRUCTURE
# ---------------------------------------------------------------------

REAL_DATA = {}
REAL_DATA_PATHS = {}
REAL_DATA_DISCOVERY = []

for dataset_name, candidates in REAL_DATA_PATH_CANDIDATES.items():
    selected_path = next(
        (
            path for path in candidates
            if path.is_file() and not section2_is_synthetic(path)
        ),
        None,
    )

    if selected_path is None:
        REAL_DATA_DISCOVERY.append({
            "dataset": dataset_name,
            "status": "MISSING",
            "path": None,
            "rows": None,
            "columns": None,
            "sha256": None,
            "details": "No approved real-data source found.",
        })
        continue

    frame = section2_read_csv(selected_path, dataset_name)

    # Prevent a one-column parse caused by an incorrect delimiter.
    if frame.shape[1] < EXPECTED_MIN_COLUMNS[dataset_name]:
        REAL_DATA_DISCOVERY.append({
            "dataset": dataset_name,
            "status": "FAIL",
            "path": str(selected_path),
            "rows": len(frame),
            "columns": len(frame.columns),
            "sha256": section2_sha256(selected_path),
            "details": (
                f"Expected at least {EXPECTED_MIN_COLUMNS[dataset_name]} "
                f"columns; parsed {len(frame.columns)}. Check delimiter "
                "and source format."
            ),
        })
        continue

    REAL_DATA[dataset_name] = frame
    REAL_DATA_PATHS[dataset_name] = selected_path

    REAL_DATA_DISCOVERY.append({
        "dataset": dataset_name,
        "status": "LOADED",
        "path": str(selected_path),
        "rows": len(frame),
        "columns": len(frame.columns),
        "sha256": section2_sha256(selected_path),
        "details": "Real-data source loaded and basic structure validated.",
    })

REAL_DATA_DISCOVERY_DF = pd.DataFrame(REAL_DATA_DISCOVERY)

REAL_DATA_DISCOVERY_PATH = (
    DIRS["inputs"] / "real_data_discovery.csv"
)
REAL_DATA_DISCOVERY_DF.to_csv(
    REAL_DATA_DISCOVERY_PATH, index=False
)

REAL_DATA_FINGERPRINT_DF = REAL_DATA_DISCOVERY_DF.loc[
    REAL_DATA_DISCOVERY_DF["status"] == "LOADED",
    ["dataset", "path", "rows", "columns", "sha256"],
].copy()

if not REAL_DATA_FINGERPRINT_DF.empty:
    REAL_DATA_FINGERPRINT_DF.insert(1, "artifact_type", "real_data")

REAL_DATA_FINGERPRINT_PATH = (
    DIRS["inputs"] / "real_data_fingerprints.csv"
)
REAL_DATA_FINGERPRINT_DF.to_csv(
    REAL_DATA_FINGERPRINT_PATH, index=False
)

display(REAL_DATA_DISCOVERY_DF)

# ---------------------------------------------------------------------
# 4. STRICT COVERAGE AND PERSISTENCE VALIDATION
# ---------------------------------------------------------------------

SECTION_2_VALIDATION_ROWS = []

for dataset_name in DATASETS:
    row = REAL_DATA_DISCOVERY_DF.loc[
        REAL_DATA_DISCOVERY_DF["dataset"] == dataset_name
    ]

    valid = (
        len(row) == 1
        and row.iloc[0]["status"] == "LOADED"
        and dataset_name in REAL_DATA
        and REAL_DATA[dataset_name].shape[1]
            >= EXPECTED_MIN_COLUMNS[dataset_name]
    )

    SECTION_2_VALIDATION_ROWS.append({
        "dataset": dataset_name,
        "status": "PASS" if valid else "FAIL",
        "path": (
            str(REAL_DATA_PATHS[dataset_name])
            if dataset_name in REAL_DATA_PATHS else None
        ),
        "rows": (
            len(REAL_DATA[dataset_name])
            if dataset_name in REAL_DATA else None
        ),
        "columns": (
            len(REAL_DATA[dataset_name].columns)
            if dataset_name in REAL_DATA else None
        ),
        "details": (
            "Real-data source and column structure validated."
            if valid
            else "Missing, malformed, or structurally invalid real-data source."
        ),
    })

SECTION_2_VALIDATION_DF = pd.DataFrame(
    SECTION_2_VALIDATION_ROWS
)

SECTION_2_VALIDATION_PATH = (
    DIRS["validation"] / "section_02_validation.csv"
)
SECTION_2_VALIDATION_DF.to_csv(
    SECTION_2_VALIDATION_PATH, index=False
)

# Reload artifacts to verify persistence.
DISCOVERY_RELOADED = pd.read_csv(REAL_DATA_DISCOVERY_PATH)
FINGERPRINT_RELOADED = pd.read_csv(REAL_DATA_FINGERPRINT_PATH)
VALIDATION_RELOADED = pd.read_csv(SECTION_2_VALIDATION_PATH)

SECTION_2_PASS = (
    set(DISCOVERY_RELOADED["dataset"]) == set(DATASETS)
    and len(FINGERPRINT_RELOADED) == len(DATASETS)
    and FINGERPRINT_RELOADED["sha256"].notna().all()
    and len(VALIDATION_RELOADED) == len(DATASETS)
    and VALIDATION_RELOADED["status"].eq("PASS").all()
)

print("=" * 80)
print("NOTEBOOK 16 — SECTION 2 VALIDATION")
print("=" * 80)
print("Expected datasets:", len(DATASETS))
print("Successfully loaded:", len(REAL_DATA))
print("Discovery artifact:", REAL_DATA_DISCOVERY_PATH)
print("Fingerprint artifact:", REAL_DATA_FINGERPRINT_PATH)
print("Validation artifact:", SECTION_2_VALIDATION_PATH)
print("Section 2 status:", "PASS" if SECTION_2_PASS else "FAIL")
print("=" * 80)

display(VALIDATION_RELOADED)

if not SECTION_2_PASS:
    raise RuntimeError(
        "Section 2 failed validation. In particular, check whether "
        "bank_marketing.csv was parsed into the expected number of columns."
    )

for dataset_name, frame in REAL_DATA.items():
    print(
        f"{dataset_name}: shape={frame.shape}, "
        f"path={REAL_DATA_PATHS[dataset_name]}"
    )

,dataset,status,path,rows,columns,sha256,details
0,adult_income,LOADED,/content/drive/MyDrive/SPP_GAN_Research/data/r...,48842,15,9479f8b76861e48c66836d97937f2c5a469581672e12c7...,Real-data source loaded and basic structure va...
1,bank_marketing,LOADED,/content/drive/MyDrive/SPP_GAN_Research/data/r...,45211,17,d1513ec63b385506f7cfce9f2c5caa9fe99e7ba4e8c3fa...,Real-data source loaded and basic structure va...
2,diabetes_130us,LOADED,/content/drive/MyDrive/SPP_GAN_Research/data/r...,101766,50,0689e7ec031237dc63031b938805c48377748761a3b26a...,Real-data source loaded and basic structure va...


NOTEBOOK 16 — SECTION 2 VALIDATION
Expected datasets: 3
Successfully loaded: 3
Discovery artifact: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/inputs/real_data_discovery.csv
Fingerprint artifact: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/inputs/real_data_fingerprints.csv
Validation artifact: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/validation/section_02_validation.csv
Section 2 status: PASS


,dataset,status,path,rows,columns,details
0,adult_income,PASS,/content/drive/MyDrive/SPP_GAN_Research/data/r...,48842,15,Real-data source and column structure validated.
1,bank_marketing,PASS,/content/drive/MyDrive/SPP_GAN_Research/data/r...,45211,17,Real-data source and column structure validated.
2,diabetes_130us,PASS,/content/drive/MyDrive/SPP_GAN_Research/data/r...,101766,50,Real-data source and column structure validated.


adult_income: shape=(48842, 15), path=/content/drive/MyDrive/SPP_GAN_Research/data/raw/adult_income.csv
bank_marketing: shape=(45211, 17), path=/content/drive/MyDrive/SPP_GAN_Research/data/raw/bank_marketing.csv
diabetes_130us: shape=(101766, 50), path=/content/drive/MyDrive/SPP_GAN_Research/data/raw/diabetes_130us.csv


In [5]:
# =====================================================================
# NOTEBOOK 16 — SECTION 3: LOAD SYNTHETIC DATA (FROZEN)
# =====================================================================

import hashlib
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------------------
# 1. Configure paths and output directories
# ---------------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Research")

NOTEBOOK_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_16"
INPUT_DIR = NOTEBOOK_ROOT / "inputs"
VALIDATION_DIR = NOTEBOOK_ROOT / "validation"

for directory in (INPUT_DIR, VALIDATION_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SYNTHETIC_DATA_PATHS = {
    "adult_income": (
        PROJECT_ROOT / "results" / "notebooks" / "notebook_13"
        / "synthetic_datasets"
        / "adult_income_synthetic_with_predicted_target.csv"
    ),
    "bank_marketing": (
        PROJECT_ROOT / "results" / "notebooks" / "notebook_13"
        / "synthetic_datasets"
        / "bank_marketing_synthetic_with_predicted_target.csv"
    ),
    "diabetes_130us": (
        PROJECT_ROOT / "results" / "notebooks" / "notebook_13"
        / "synthetic_datasets"
        / "diabetes_130us_synthetic_with_predicted_target.csv"
    ),
}

# Expected shapes are based on the observed canonical Notebook 13 artifacts.
# Schema alignment with real data is evaluated separately in Section 4.
EXPECTED_SYNTHETIC_COLUMNS = {
    "adult_income": 15,
    "bank_marketing": 17,
    "diabetes_130us": 48,
}

EXPECTED_DATASETS = set(SYNTHETIC_DATA_PATHS)

# ---------------------------------------------------------------------
# 2. Define validation utilities
# ---------------------------------------------------------------------

def section3_sha256(file_path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def section3_is_synthetic_path(file_path):
    parts = [part.lower() for part in file_path.parts]
    filename = file_path.name.lower()

    return (
        "synthetic" in parts
        or "synthetic" in filename
        or "generated" in filename
    )


# ---------------------------------------------------------------------
# 3. Load only the explicitly configured synthetic artifacts
# ---------------------------------------------------------------------

SYNTHETIC_DATA = {}
synthetic_records = []
fingerprint_records = []
validation_records = []

for dataset_name, file_path in SYNTHETIC_DATA_PATHS.items():
    frame = None
    file_hash = None
    status = "FAIL"
    details = ""

    try:
        if not file_path.is_file():
            details = "Configured canonical synthetic artifact does not exist."
        elif not section3_is_synthetic_path(file_path):
            details = "Configured path does not identify a synthetic artifact."
        else:
            frame = pd.read_csv(file_path, low_memory=False)

            if frame.empty:
                details = "Synthetic artifact contains no rows."
            elif frame.columns.duplicated().any():
                details = "Synthetic artifact contains duplicate column names."
            elif len(frame.columns) != EXPECTED_SYNTHETIC_COLUMNS[dataset_name]:
                details = (
                    f"Unexpected column count: {len(frame.columns)}; "
                    f"expected {EXPECTED_SYNTHETIC_COLUMNS[dataset_name]}."
                )
            else:
                file_hash = section3_sha256(file_path)
                SYNTHETIC_DATA[dataset_name] = frame
                status = "PASS"
                details = "Canonical synthetic artifact loaded and validated."

    except Exception as exc:
        details = f"{type(exc).__name__}: {exc}"

    synthetic_records.append({
        "dataset": dataset_name,
        "status": "LOADED" if frame is not None and not frame.empty else "FAIL",
        "path": str(file_path),
        "rows": len(frame) if frame is not None else None,
        "columns": len(frame.columns) if frame is not None else None,
        "sha256": file_hash,
        "details": details,
    })

    fingerprint_records.append({
        "dataset": dataset_name,
        "path": str(file_path),
        "sha256": file_hash,
        "rows": len(frame) if frame is not None else None,
        "columns": len(frame.columns) if frame is not None else None,
    })

    validation_records.append({
        "dataset": dataset_name,
        "status": status,
        "path": str(file_path),
        "rows": len(frame) if frame is not None else None,
        "columns": len(frame.columns) if frame is not None else None,
        "expected_columns": EXPECTED_SYNTHETIC_COLUMNS[dataset_name],
        "details": details,
    })

# ---------------------------------------------------------------------
# 4. Save discovery, fingerprint, and validation artifacts
# ---------------------------------------------------------------------

SYNTHETIC_DATA_DISCOVERY_DF = pd.DataFrame(synthetic_records)
SYNTHETIC_DATA_FINGERPRINTS_DF = pd.DataFrame(fingerprint_records)
SECTION_3_VALIDATION_DF = pd.DataFrame(validation_records)

discovery_path = INPUT_DIR / "synthetic_data_discovery.csv"
fingerprint_path = INPUT_DIR / "synthetic_data_fingerprints.csv"
validation_path = VALIDATION_DIR / "section_03_validation.csv"

SYNTHETIC_DATA_DISCOVERY_DF.to_csv(discovery_path, index=False)
SYNTHETIC_DATA_FINGERPRINTS_DF.to_csv(fingerprint_path, index=False)
SECTION_3_VALIDATION_DF.to_csv(validation_path, index=False)

# ---------------------------------------------------------------------
# 5. Reload and verify persisted artifacts
# ---------------------------------------------------------------------

DISCOVERY_RELOADED = pd.read_csv(discovery_path)
FINGERPRINTS_RELOADED = pd.read_csv(fingerprint_path)
VALIDATION_RELOADED = pd.read_csv(validation_path)

assert len(DISCOVERY_RELOADED) == len(EXPECTED_DATASETS), (
    "Discovery artifact row count mismatch."
)
assert len(FINGERPRINTS_RELOADED) == len(EXPECTED_DATASETS), (
    "Fingerprint artifact row count mismatch."
)
assert len(VALIDATION_RELOADED) == len(EXPECTED_DATASETS), (
    "Validation artifact row count mismatch."
)

assert set(DISCOVERY_RELOADED["dataset"]) == EXPECTED_DATASETS, (
    "Discovery artifact dataset coverage mismatch."
)

assert set(SYNTHETIC_DATA) == EXPECTED_DATASETS, (
    "Not all configured synthetic datasets passed validation."
)

assert (VALIDATION_RELOADED["status"] == "PASS").all(), (
    "At least one synthetic dataset failed validation."
)

# Verify every loaded dataset's saved fingerprint against the source file.
for dataset_name, file_path in SYNTHETIC_DATA_PATHS.items():
    saved_hash = FINGERPRINTS_RELOADED.loc[
        FINGERPRINTS_RELOADED["dataset"] == dataset_name, "sha256"
    ].iloc[0]

    assert isinstance(saved_hash, str) and saved_hash, (
        f"Missing saved fingerprint for {dataset_name}."
    )
    assert section3_sha256(file_path) == saved_hash, (
        f"Fingerprint mismatch for {dataset_name}."
    )

# ---------------------------------------------------------------------
# 6. Report completion
# ---------------------------------------------------------------------

SECTION_3_PASS = True

print("=" * 80)
print("NOTEBOOK 16 — SECTION 3: LOAD SYNTHETIC DATA")
print("=" * 80)

display(SYNTHETIC_DATA_DISCOVERY_DF[
    ["dataset", "status", "path", "rows", "columns", "sha256", "details"]
])

print("\n" + "=" * 80)
print("NOTEBOOK 16 — SECTION 3 VALIDATION")
print("=" * 80)
print(f"Expected datasets: {len(EXPECTED_DATASETS)}")
print(f"Successfully loaded: {len(SYNTHETIC_DATA)}")
print(f"Discovery artifact: {discovery_path}")
print(f"Fingerprint artifact: {fingerprint_path}")
print(f"Validation artifact: {validation_path}")
print(f"Section 3 status: {'PASS' if SECTION_3_PASS else 'FAIL'}")
print("=" * 80)

display(SECTION_3_VALIDATION_DF)

NOTEBOOK 16 — SECTION 3: LOAD SYNTHETIC DATA


,dataset,status,path,rows,columns,sha256,details
0,adult_income,LOADED,/content/drive/MyDrive/SPP_GAN_Research/result...,34189,15,8db2fff4a874767e2620a39e2081b71edd06226d6f2065...,Canonical synthetic artifact loaded and valida...
1,bank_marketing,LOADED,/content/drive/MyDrive/SPP_GAN_Research/result...,31647,17,ba2825bf7e372d64c5c5b14d70700305e4d88617532b6f...,Canonical synthetic artifact loaded and valida...
2,diabetes_130us,LOADED,/content/drive/MyDrive/SPP_GAN_Research/result...,71236,48,d8ccfde666814d01e740e8cef0338639b73ef21b2d2a38...,Canonical synthetic artifact loaded and valida...



NOTEBOOK 16 — SECTION 3 VALIDATION
Expected datasets: 3
Successfully loaded: 3
Discovery artifact: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/inputs/synthetic_data_discovery.csv
Fingerprint artifact: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/inputs/synthetic_data_fingerprints.csv
Validation artifact: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/validation/section_03_validation.csv
Section 3 status: PASS


,dataset,status,path,rows,columns,expected_columns,details
0,adult_income,PASS,/content/drive/MyDrive/SPP_GAN_Research/result...,34189,15,15,Canonical synthetic artifact loaded and valida...
1,bank_marketing,PASS,/content/drive/MyDrive/SPP_GAN_Research/result...,31647,17,17,Canonical synthetic artifact loaded and valida...
2,diabetes_130us,PASS,/content/drive/MyDrive/SPP_GAN_Research/result...,71236,48,48,Canonical synthetic artifact loaded and valida...


In [6]:
# =====================================================================
# NOTEBOOK 16 — SECTION 4: VALIDATE ALIGNMENT (FROZEN)
# =====================================================================

import json
import pandas as pd

ALIGNMENT_ROWS = []
ALIGNED_REAL = {}
ALIGNED_SYNTHETIC = {}

REQUIRED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

# Columns omitted from the diabetes synthetic artifact by Notebook 13.
# Do not silently drop these from the real dataset: they must be excluded
# from both sides explicitly for a valid like-for-like privacy comparison.
EXPLICIT_EXCLUDED_COLUMNS = {
    "adult_income": [],
    "bank_marketing": [],
    "diabetes_130us": ["encounter_id", "patient_nbr"],
}

for dataset_name in REQUIRED_DATASETS:
    if dataset_name not in REAL_DATA:
        raise KeyError(f"Real dataset missing: {dataset_name}")

    if dataset_name not in SYNTHETIC_DATA:
        raise KeyError(f"Synthetic dataset missing: {dataset_name}")

    real_df = REAL_DATA[dataset_name].copy()
    synth_df = SYNTHETIC_DATA[dataset_name].copy()

    real_cols = list(real_df.columns)
    synth_cols = list(synth_df.columns)

    missing_in_synthetic = [
        col for col in real_cols if col not in synth_cols
    ]
    extra_in_synthetic = [
        col for col in synth_cols if col not in real_cols
    ]

    excluded_cols = EXPLICIT_EXCLUDED_COLUMNS[dataset_name]

    # Confirm that every declared exclusion is present in real data,
    # absent from synthetic data, and explicitly recorded.
    exclusions_valid = all(
        col in real_cols and col not in synth_cols
        for col in excluded_cols
    )

    unexplained_missing = [
        col for col in missing_in_synthetic
        if col not in excluded_cols
    ]

    schema_pass = (
        not extra_in_synthetic
        and not unexplained_missing
        and exclusions_valid
    )

    ALIGNMENT_ROWS.append({
        "dataset": dataset_name,
        "real_rows": len(real_df),
        "synthetic_rows": len(synth_df),
        "real_columns": len(real_cols),
        "synthetic_columns": len(synth_cols),
        "missing_in_synthetic": json.dumps(missing_in_synthetic),
        "extra_in_synthetic": json.dumps(extra_in_synthetic),
        "explicitly_excluded_columns": json.dumps(excluded_cols),
        "unexplained_missing_columns": json.dumps(unexplained_missing),
        "exclusions_valid": exclusions_valid,
        "schema_status": "PASS" if schema_pass else "FAIL",
    })

    if not schema_pass:
        continue

    # Align only after the schema differences have been explicitly
    # explained. Preserve synthetic column order for both datasets.
    comparison_cols = list(synth_cols)

    real_aligned = real_df[comparison_cols].copy()
    synth_aligned = synth_df[comparison_cols].copy()

    ALIGNED_REAL[dataset_name] = real_aligned.reset_index(drop=True)
    ALIGNED_SYNTHETIC[dataset_name] = synth_aligned.reset_index(drop=True)

# ---------------------------------------------------------------------
# Save and validate alignment artifacts
# ---------------------------------------------------------------------

ALIGNMENT_DF = pd.DataFrame(ALIGNMENT_ROWS)

alignment_path = DIRS["inputs"] / "alignment_validation.csv"
ALIGNMENT_DF.to_csv(alignment_path, index=False)

display(ALIGNMENT_DF)

failed_alignment = ALIGNMENT_DF.loc[
    ALIGNMENT_DF["schema_status"] != "PASS", "dataset"
].tolist()

if failed_alignment:
    raise ValueError(
        "Real/synthetic schema alignment failed for: "
        + ", ".join(failed_alignment)
        + ". Resolve the schema provenance before privacy evaluation."
    )

assert set(ALIGNED_REAL) == set(REQUIRED_DATASETS)
assert set(ALIGNED_SYNTHETIC) == set(REQUIRED_DATASETS)

for dataset_name in REQUIRED_DATASETS:
    assert list(ALIGNED_REAL[dataset_name].columns) == list(
        ALIGNED_SYNTHETIC[dataset_name].columns
    ), f"Aligned column order mismatch: {dataset_name}"

    assert ALIGNED_REAL[dataset_name].shape[1] == (
        ALIGNED_SYNTHETIC[dataset_name].shape[1]
    ), f"Aligned column count mismatch: {dataset_name}"

print(f"Alignment artifact: {alignment_path}")
print("Schema alignment: PASS")

,dataset,real_rows,synthetic_rows,real_columns,synthetic_columns,missing_in_synthetic,extra_in_synthetic,explicitly_excluded_columns,unexplained_missing_columns,exclusions_valid,schema_status
0,adult_income,48842,34189,15,15,[],[],[],[],True,PASS
1,bank_marketing,45211,31647,17,17,[],[],[],[],True,PASS
2,diabetes_130us,101766,71236,50,48,"[""encounter_id"", ""patient_nbr""]",[],"[""encounter_id"", ""patient_nbr""]",[],True,PASS


Alignment artifact: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/inputs/alignment_validation.csv
Schema alignment: PASS


In [8]:
# =====================================================================
# NOTEBOOK 16 — SECTION 5: EXACT DUPLICATE ANALYSIS (FROZEN)
# =====================================================================

import hashlib
import json
import numpy as np
import pandas as pd
from pathlib import Path

EXACT_DUPLICATE_ROWS = []
EXACT_DUPLICATE_DETAILS = []

# Use SHA-256 over a deterministic, type-aware row serialization.
# Hashes are used as candidate matches; full canonical rows are compared
# to avoid treating hash equality alone as proof of a duplicate.

MISSING_TOKEN = "<MISSING_VALUE>"


def canonicalize_for_hash(df):
    """Create deterministic, type-aware representations of row values."""
    canonical = pd.DataFrame(index=df.index)

    for column in df.columns:
        series = df[column]

        def normalize(value):
            if pd.isna(value):
                return ["missing", None]
            return [
                type(value).__name__,
                str(value)
            ]

        canonical[column] = series.map(normalize)

    return canonical


def canonical_row_payload(row_values):
    """Serialize one normalized row deterministically."""
    return json.dumps(
        row_values,
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")


def row_hashes(df):
    """Return SHA-256 digests and canonical row payloads."""
    canonical = canonicalize_for_hash(df)
    payloads = [
        canonical_row_payload(row)
        for row in canonical.itertuples(index=False, name=None)
    ]
    hashes = [
        hashlib.sha256(payload).hexdigest()
        for payload in payloads
    ]
    return hashes, payloads


for dataset_name in DATASETS:
    real_df = ALIGNED_REAL[dataset_name]
    synth_df = ALIGNED_SYNTHETIC[dataset_name]

    if list(real_df.columns) != list(synth_df.columns):
        raise ValueError(
            f"Column order mismatch before exact duplicate analysis: "
            f"{dataset_name}"
        )

    real_hashes, real_payloads = row_hashes(real_df)
    synth_hashes, synth_payloads = row_hashes(synth_df)

    # Keep payloads for collision-safe equality verification.
    real_payloads_by_hash = {}
    for digest, payload in zip(real_hashes, real_payloads):
        real_payloads_by_hash.setdefault(digest, set()).add(payload)

    matched_indices = []
    matched_hashes = []
    collision_checks_passed = True

    for index, (digest, payload) in enumerate(
        zip(synth_hashes, synth_payloads)
    ):
        candidate_payloads = real_payloads_by_hash.get(digest, set())

        if payload in candidate_payloads:
            matched_indices.append(index)
            matched_hashes.append(digest)
        elif candidate_payloads:
            # A matching digest but different payload indicates a collision.
            collision_checks_passed = False

    matched_synthetic_count = len(matched_indices)
    unique_matched_hashes = len(set(matched_hashes))

    exact_rate = (
        matched_synthetic_count / len(synth_df)
        if len(synth_df) else np.nan
    )

    status = "PASS" if collision_checks_passed else "FAIL"

    EXACT_DUPLICATE_ROWS.append({
        "dataset": dataset_name,
        "real_rows": len(real_df),
        "synthetic_rows": len(synth_df),
        "synthetic_rows_matching_real": matched_synthetic_count,
        "unique_matching_synthetic_row_hashes": unique_matched_hashes,
        "exact_duplicate_rate": exact_rate,
        "status": status,
        "method": "SHA-256 of deterministic type-aware canonical rows",
        "columns_compared": json.dumps(list(real_df.columns)),
    })

    if matched_indices:
        matched_rows = synth_df.iloc[matched_indices].copy()
        matched_rows.insert(0, "synthetic_row_index", matched_indices)
        matched_rows.to_csv(
            DIRS["exact"] / f"{dataset_name}_exact_matches.csv",
            index=False,
        )

    EXACT_DUPLICATE_DETAILS.append({
        "dataset": dataset_name,
        "matched_synthetic_indices": json.dumps(matched_indices),
    })

EXACT_DUPLICATE_DF = pd.DataFrame(EXACT_DUPLICATE_ROWS)
EXACT_DUPLICATE_DETAILS_DF = pd.DataFrame(EXACT_DUPLICATE_DETAILS)

summary_path = DIRS["exact"] / "exact_duplicate_summary.csv"
details_path = DIRS["exact"] / "exact_duplicate_details.csv"

EXACT_DUPLICATE_DF.to_csv(summary_path, index=False)
EXACT_DUPLICATE_DETAILS_DF.to_csv(details_path, index=False)

# Reload and verify saved summaries.
EXACT_DUPLICATE_RELOADED = pd.read_csv(summary_path)
EXACT_DETAILS_RELOADED = pd.read_csv(details_path)

assert len(EXACT_DUPLICATE_RELOADED) == len(DATASETS)
assert len(EXACT_DETAILS_RELOADED) == len(DATASETS)
assert set(EXACT_DUPLICATE_RELOADED["dataset"]) == set(DATASETS)
assert (EXACT_DUPLICATE_RELOADED["status"] == "PASS").all()

display(EXACT_DUPLICATE_DF)

print(f"Summary saved: {summary_path}")
print(f"Details saved: {details_path}")
print("Exact duplicate analysis validation: PASS")

,dataset,real_rows,synthetic_rows,synthetic_rows_matching_real,unique_matching_synthetic_row_hashes,exact_duplicate_rate,status,method,columns_compared
0,adult_income,48842,34189,0,0,0.0,PASS,SHA-256 of deterministic type-aware canonical ...,"[""age"", ""workclass"", ""fnlwgt"", ""education"", ""e..."
1,bank_marketing,45211,31647,0,0,0.0,PASS,SHA-256 of deterministic type-aware canonical ...,"[""age"", ""job"", ""marital"", ""education"", ""defaul..."
2,diabetes_130us,101766,71236,0,0,0.0,PASS,SHA-256 of deterministic type-aware canonical ...,"[""race"", ""gender"", ""age"", ""weight"", ""admission..."


Summary saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/exact_duplicate_analysis/exact_duplicate_summary.csv
Details saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/exact_duplicate_analysis/exact_duplicate_details.csv
Exact duplicate analysis validation: PASS


In [10]:
# =====================================================================
# NOTEBOOK 16 — SECTION 6: NEAR-DUPLICATE ANALYSIS (FROZEN)
# =====================================================================

import gc
import json
import numpy as np
import pandas as pd

NEAR_DUPLICATE_ROWS = []
NEAR_DUPLICATE_SCORE_FILES = []

MISSING_TOKEN = "<MISSING_VALUE>"


def infer_feature_types(real_df):
    """Infer numeric and categorical columns from the real reference schema."""
    numeric_cols = real_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [
        col for col in real_df.columns if col not in numeric_cols
    ]
    return numeric_cols, categorical_cols


def prepare_similarity_schema(real_df):
    """Build feature ranges using real data only."""
    numeric_cols, categorical_cols = infer_feature_types(real_df)
    numeric_ranges = {}

    for col in numeric_cols:
        values = pd.to_numeric(real_df[col], errors="coerce").to_numpy(dtype=float)
        finite_values = values[np.isfinite(values)]

        if len(finite_values):
            low = float(finite_values.min())
            high = float(finite_values.max())
            numeric_ranges[col] = max(high - low, 1e-12)
        else:
            numeric_ranges[col] = 1.0

    return numeric_cols, categorical_cols, numeric_ranges


def sample_rows(df, max_rows, seed):
    """Select a reproducible sample without modifying the source dataframe."""
    if len(df) <= max_rows:
        return df.copy().reset_index(drop=True)

    return df.sample(
        n=max_rows, random_state=seed
    ).reset_index(drop=True)


def mixed_similarity_block(query_df, reference_df, schema):
    """
    Compute pairwise mixed-type similarity.

    Numeric feature similarity is range-normalized.
    Categorical feature similarity is exact agreement.
    A missing value matches another missing value, but not a nonmissing value.
    """
    numeric_cols, categorical_cols, numeric_ranges = schema

    n_query = len(query_df)
    n_reference = len(reference_df)

    if list(query_df.columns) != list(reference_df.columns):
        raise ValueError("Query and reference schemas are not aligned.")

    similarity_sum = np.zeros(
        (n_query, n_reference), dtype=np.float32
    )
    feature_count = np.zeros(
        (n_query, n_reference), dtype=np.float32
    )

    for col in numeric_cols:
        q = pd.to_numeric(
            query_df[col], errors="coerce"
        ).to_numpy(dtype=float)

        r = pd.to_numeric(
            reference_df[col], errors="coerce"
        ).to_numpy(dtype=float)

        q_missing = ~np.isfinite(q)
        r_missing = ~np.isfinite(r)

        # Replace nonfinite values before subtraction to avoid invalid
        # arithmetic warnings; missingness is handled explicitly below.
        q_safe = np.where(q_missing, 0.0, q)
        r_safe = np.where(r_missing, 0.0, r)

        difference = (
            np.abs(q_safe[:, None] - r_safe[None, :])
            / numeric_ranges[col]
        )

        feature_sim = 1.0 - np.clip(difference, 0.0, 1.0)

        both_missing = q_missing[:, None] & r_missing[None, :]
        one_missing = q_missing[:, None] ^ r_missing[None, :]

        feature_sim[both_missing] = 1.0
        feature_sim[one_missing] = 0.0
        feature_sim[~np.isfinite(feature_sim)] = 0.0

        similarity_sum += feature_sim.astype(np.float32)
        feature_count += 1.0

    for col in categorical_cols:
        q = (
            query_df[col]
            .astype("string")
            .fillna(MISSING_TOKEN)
            .to_numpy()
        )
        r = (
            reference_df[col]
            .astype("string")
            .fillna(MISSING_TOKEN)
            .to_numpy()
        )

        feature_sim = (q[:, None] == r[None, :]).astype(np.float32)

        similarity_sum += feature_sim
        feature_count += 1.0

    # A schema with no usable columns has no meaningful similarity.
    if not numeric_cols and not categorical_cols:
        raise ValueError("No features available for similarity analysis.")

    feature_count[feature_count == 0] = 1.0
    return similarity_sum / feature_count


# ---------------------------------------------------------------------
# 1. Validate configuration
# ---------------------------------------------------------------------

if MAX_REAL_REFERENCE_ROWS < 1 or MAX_SYNTHETIC_ROWS < 1:
    raise ValueError("Sampling limits must be positive integers.")

if PAIRWISE_CHUNK_SIZE < 1:
    raise ValueError("PAIRWISE_CHUNK_SIZE must be a positive integer.")

if not 0.0 <= NEAR_DUPLICATE_THRESHOLD <= 1.0:
    raise ValueError("NEAR_DUPLICATE_THRESHOLD must be between 0 and 1.")

# ---------------------------------------------------------------------
# 2. Run deterministic near-duplicate analysis
# ---------------------------------------------------------------------

for dataset_name in DATASETS:
    real_df = ALIGNED_REAL[dataset_name]
    synth_df = ALIGNED_SYNTHETIC[dataset_name]

    if list(real_df.columns) != list(synth_df.columns):
        raise ValueError(
            f"Column order mismatch before similarity analysis: {dataset_name}"
        )

    if real_df.empty or synth_df.empty:
        raise ValueError(
            f"Empty real or synthetic dataset: {dataset_name}"
        )

    schema = prepare_similarity_schema(real_df)

    real_reference = sample_rows(
        real_df, MAX_REAL_REFERENCE_ROWS, PRIVACY_SEED
    )
    synth_sample = sample_rows(
        synth_df, MAX_SYNTHETIC_ROWS, PRIVACY_SEED + 1
    )

    nearest_similarity = np.full(
        len(synth_sample), np.nan, dtype=np.float32
    )

    for start in range(0, len(synth_sample), PAIRWISE_CHUNK_SIZE):
        end = min(
            start + PAIRWISE_CHUNK_SIZE, len(synth_sample)
        )

        block = mixed_similarity_block(
            synth_sample.iloc[start:end],
            real_reference,
            schema,
        )

        nearest_similarity[start:end] = block.max(axis=1)

        del block
        gc.collect()

    if not np.isfinite(nearest_similarity).all():
        raise ValueError(
            f"Nonfinite nearest-neighbor similarity detected: {dataset_name}"
        )

    near_mask = nearest_similarity >= NEAR_DUPLICATE_THRESHOLD

    score_df = pd.DataFrame({
        "sampled_synthetic_row_index": np.arange(len(synth_sample)),
        "nearest_real_similarity": nearest_similarity,
        "near_duplicate_at_threshold": near_mask,
    })

    score_path = (
        DIRS["near"] / f"{dataset_name}_near_duplicate_scores.csv"
    )
    score_df.to_csv(score_path, index=False)

    NEAR_DUPLICATE_SCORE_FILES.append({
        "dataset": dataset_name,
        "path": str(score_path),
        "rows_saved": len(score_df),
    })

    NEAR_DUPLICATE_ROWS.append({
        "dataset": dataset_name,
        "synthetic_rows_loaded": len(synth_df),
        "synthetic_rows_sampled": len(synth_sample),
        "real_reference_rows_sampled": len(real_reference),
        "similarity_threshold": NEAR_DUPLICATE_THRESHOLD,
        "near_duplicate_count_in_sample": int(near_mask.sum()),
        "near_duplicate_rate_in_sample": float(near_mask.mean()),
        "nearest_similarity_mean": float(nearest_similarity.mean()),
        "nearest_similarity_median": float(np.median(nearest_similarity)),
        "nearest_similarity_p95": float(
            np.quantile(nearest_similarity, 0.95)
        ),
        "status": "PASS",
        "sampling_method": "Deterministic row sampling",
        "similarity_method": (
            "Range-normalized numeric similarity and categorical agreement"
        ),
        "score_file": str(score_path),
    })

    del real_reference, synth_sample, nearest_similarity, score_df
    gc.collect()

# ---------------------------------------------------------------------
# 3. Save and verify summary artifacts
# ---------------------------------------------------------------------

NEAR_DUPLICATE_DF = pd.DataFrame(NEAR_DUPLICATE_ROWS)
NEAR_DUPLICATE_FILES_DF = pd.DataFrame(NEAR_DUPLICATE_SCORE_FILES)

summary_path = DIRS["near"] / "near_duplicate_summary.csv"
files_path = DIRS["near"] / "near_duplicate_score_files.csv"

NEAR_DUPLICATE_DF.to_csv(summary_path, index=False)
NEAR_DUPLICATE_FILES_DF.to_csv(files_path, index=False)

NEAR_DUPLICATE_RELOADED = pd.read_csv(summary_path)
NEAR_DUPLICATE_FILES_RELOADED = pd.read_csv(files_path)

assert len(NEAR_DUPLICATE_RELOADED) == len(DATASETS)
assert len(NEAR_DUPLICATE_FILES_RELOADED) == len(DATASETS)
assert set(NEAR_DUPLICATE_RELOADED["dataset"]) == set(DATASETS)
assert (NEAR_DUPLICATE_RELOADED["status"] == "PASS").all()

for record in NEAR_DUPLICATE_FILES_RELOADED.to_dict("records"):
    score_path = record["path"]
    saved_scores = pd.read_csv(score_path)

    assert len(saved_scores) == int(record["rows_saved"])
    assert saved_scores["nearest_real_similarity"].between(
        0.0, 1.0
    ).all()

display(NEAR_DUPLICATE_DF)

print(f"Summary saved: {summary_path}")
print(f"Score-file registry saved: {files_path}")
print("Near-duplicate analysis validation: PASS")

,dataset,synthetic_rows_loaded,synthetic_rows_sampled,real_reference_rows_sampled,similarity_threshold,near_duplicate_count_in_sample,near_duplicate_rate_in_sample,nearest_similarity_mean,nearest_similarity_median,nearest_similarity_p95,status,sampling_method,similarity_method,score_file
0,adult_income,34189,5000,5000,0.95,1668,0.3336,0.919375,0.915252,0.983658,PASS,Deterministic row sampling,Range-normalized numeric similarity and catego...,/content/drive/MyDrive/SPP_GAN_Research/result...
1,bank_marketing,31647,5000,5000,0.95,1306,0.2612,0.925582,0.924511,0.985996,PASS,Deterministic row sampling,Range-normalized numeric similarity and catego...,/content/drive/MyDrive/SPP_GAN_Research/result...
2,diabetes_130us,71236,5000,5000,0.95,0,0.0000,0.820498,0.820714,0.846390,PASS,Deterministic row sampling,Range-normalized numeric similarity and catego...,/content/drive/MyDrive/SPP_GAN_Research/result...


Summary saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/near_duplicate_analysis/near_duplicate_summary.csv
Score-file registry saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/near_duplicate_analysis/near_duplicate_score_files.csv
Near-duplicate analysis validation: PASS


In [12]:
# =====================================================================
# NOTEBOOK 16 — SECTION 7: NEAREST-NEIGHBOR / DISTANCE RISK (FROZEN)
# =====================================================================

import gc
import numpy as np
import pandas as pd

DISTANCE_RISK_ROWS = []
DISTANCE_RISK_FILES = []

for dataset_name in DATASETS:
    real_df = ALIGNED_REAL[dataset_name]
    synth_df = ALIGNED_SYNTHETIC[dataset_name]

    if list(real_df.columns) != list(synth_df.columns):
        raise ValueError(
            f"Column order mismatch for {dataset_name}."
        )

    if len(real_df) < 2 or len(synth_df) < 1:
        raise ValueError(
            f"Insufficient rows for nearest-neighbor analysis: {dataset_name}"
        )

    schema = prepare_similarity_schema(real_df)

    real_reference = sample_rows(
        real_df, MAX_REAL_REFERENCE_ROWS, PRIVACY_SEED + 10
    )
    synth_sample = sample_rows(
        synth_df, MAX_SYNTHETIC_ROWS, PRIVACY_SEED + 11
    )

    if len(real_reference) < 2:
        raise ValueError(
            f"At least two real reference rows are required: {dataset_name}"
        )

    # ---------------------------------------------------------------
    # 1. Synthetic -> nearest sampled real record
    # ---------------------------------------------------------------

    synthetic_min_distance = np.full(
        len(synth_sample), np.nan, dtype=np.float32
    )

    for start in range(0, len(synth_sample), PAIRWISE_CHUNK_SIZE):
        end = min(
            start + PAIRWISE_CHUNK_SIZE, len(synth_sample)
        )

        sim = mixed_similarity_block(
            synth_sample.iloc[start:end],
            real_reference,
            schema,
        )

        synthetic_min_distance[start:end] = 1.0 - sim.max(axis=1)

        del sim
        gc.collect()

    # ---------------------------------------------------------------
    # 2. Real -> nearest OTHER sampled real record
    # ---------------------------------------------------------------

    real_min_distance = np.full(
        len(real_reference), np.nan, dtype=np.float32
    )

    for start in range(0, len(real_reference), PAIRWISE_CHUNK_SIZE):
        end = min(
            start + PAIRWISE_CHUNK_SIZE, len(real_reference)
        )

        sim = mixed_similarity_block(
            real_reference.iloc[start:end],
            real_reference,
            schema,
        )

        # Exclude each query's own reference row by position.
        for local_i, global_i in enumerate(range(start, end)):
            sim[local_i, global_i] = -np.inf

        real_min_distance[start:end] = 1.0 - sim.max(axis=1)

        del sim
        gc.collect()

    if not np.isfinite(synthetic_min_distance).all():
        raise ValueError(
            f"Nonfinite synthetic distances for {dataset_name}."
        )

    valid_real_distances = real_min_distance[
        np.isfinite(real_min_distance)
    ]

    if len(valid_real_distances) != len(real_reference):
        raise ValueError(
            f"Invalid real nearest-neighbor distances for {dataset_name}."
        )

    # ---------------------------------------------------------------
    # 3. Summarize distance distributions and proximity indicator
    # ---------------------------------------------------------------

    reference_p05 = float(
        np.quantile(valid_real_distances, 0.05)
    )
    reference_median = float(
        np.median(valid_real_distances)
    )
    reference_p95 = float(
        np.quantile(valid_real_distances, 0.95)
    )

    close_threshold_rate = float(
        np.mean(synthetic_min_distance < reference_p05)
    )

    DISTANCE_RISK_ROWS.append({
        "dataset": dataset_name,
        "synthetic_sample_rows": len(synth_sample),
        "real_reference_rows": len(real_reference),
        "synthetic_nn_distance_mean": float(
            np.mean(synthetic_min_distance)
        ),
        "synthetic_nn_distance_median": float(
            np.median(synthetic_min_distance)
        ),
        "synthetic_nn_distance_p05": float(
            np.quantile(synthetic_min_distance, 0.05)
        ),
        "synthetic_nn_distance_p95": float(
            np.quantile(synthetic_min_distance, 0.95)
        ),
        "real_nn_distance_p05": reference_p05,
        "real_nn_distance_median": reference_median,
        "real_nn_distance_p95": reference_p95,
        "synthetic_below_real_nn_p05_rate": close_threshold_rate,
        "interpretation": (
            "Empirical proximity indicator; not a formal privacy guarantee"
        ),
        "status": "PASS",
    })

    # ---------------------------------------------------------------
    # 4. Persist per-row distance artifacts
    # ---------------------------------------------------------------

    synthetic_distance_path = (
        DIRS["distance"]
        / f"{dataset_name}_synthetic_nn_distances.csv"
    )
    real_distance_path = (
        DIRS["distance"]
        / f"{dataset_name}_real_reference_nn_distances.csv"
    )

    pd.DataFrame({
        "sampled_synthetic_row_index": np.arange(len(synth_sample)),
        "synthetic_nearest_real_distance": synthetic_min_distance,
    }).to_csv(synthetic_distance_path, index=False)

    pd.DataFrame({
        "sampled_real_row_index": np.arange(len(real_reference)),
        "real_nearest_other_real_distance": real_min_distance,
    }).to_csv(real_distance_path, index=False)

    DISTANCE_RISK_FILES.extend([
        {
            "dataset": dataset_name,
            "artifact_type": "synthetic_nearest_real_distances",
            "path": str(synthetic_distance_path),
            "rows_saved": len(synth_sample),
        },
        {
            "dataset": dataset_name,
            "artifact_type": "real_nearest_other_real_distances",
            "path": str(real_distance_path),
            "rows_saved": len(real_reference),
        },
    ])

    del real_reference, synth_sample
    del synthetic_min_distance, real_min_distance, valid_real_distances
    gc.collect()

# ---------------------------------------------------------------------
# 5. Save summaries and artifact registry
# ---------------------------------------------------------------------

DISTANCE_RISK_DF = pd.DataFrame(DISTANCE_RISK_ROWS)
DISTANCE_RISK_FILES_DF = pd.DataFrame(DISTANCE_RISK_FILES)

summary_path = (
    DIRS["distance"] / "nearest_neighbor_risk_summary.csv"
)
registry_path = (
    DIRS["distance"] / "nearest_neighbor_distance_files.csv"
)

DISTANCE_RISK_DF.to_csv(summary_path, index=False)
DISTANCE_RISK_FILES_DF.to_csv(registry_path, index=False)

# ---------------------------------------------------------------------
# 6. Reload and validate saved outputs
# ---------------------------------------------------------------------

DISTANCE_RISK_RELOADED = pd.read_csv(summary_path)
DISTANCE_FILES_RELOADED = pd.read_csv(registry_path)

assert len(DISTANCE_RISK_RELOADED) == len(DATASETS)
assert set(DISTANCE_RISK_RELOADED["dataset"]) == set(DATASETS)
assert (DISTANCE_RISK_RELOADED["status"] == "PASS").all()
assert len(DISTANCE_FILES_RELOADED) == 2 * len(DATASETS)

for record in DISTANCE_FILES_RELOADED.to_dict("records"):
    saved_df = pd.read_csv(record["path"])

    assert len(saved_df) == int(record["rows_saved"])
    assert saved_df.shape[0] > 0

    distance_column = (
        "synthetic_nearest_real_distance"
        if record["artifact_type"] == "synthetic_nearest_real_distances"
        else "real_nearest_other_real_distance"
    )

    assert saved_df[distance_column].between(0.0, 1.0).all(), (
        f"Invalid distance values in {record['path']}"
    )

display(DISTANCE_RISK_DF)

print(f"Summary saved: {summary_path}")
print(f"Artifact registry saved: {registry_path}")
print("Nearest-neighbor distance validation: PASS")

,dataset,synthetic_sample_rows,real_reference_rows,synthetic_nn_distance_mean,synthetic_nn_distance_median,synthetic_nn_distance_p05,synthetic_nn_distance_p95,real_nn_distance_p05,real_nn_distance_median,real_nn_distance_p95,synthetic_below_real_nn_p05_rate,interpretation,status
0,adult_income,5000,5000,0.079478,0.084399,0.016032,0.161644,0.002683,0.022561,0.142559,0.0000,Empirical proximity indicator; not a formal pr...,PASS
1,bank_marketing,5000,5000,0.074965,0.075786,0.013910,0.140404,0.003652,0.018386,0.093631,0.0004,Empirical proximity indicator; not a formal pr...,PASS
2,diabetes_130us,5000,5000,0.178350,0.178375,0.150601,0.205914,0.063778,0.098182,0.140067,0.0000,Empirical proximity indicator; not a formal pr...,PASS


Summary saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/nearest_neighbor_analysis/nearest_neighbor_risk_summary.csv
Artifact registry saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/nearest_neighbor_analysis/nearest_neighbor_distance_files.csv
Nearest-neighbor distance validation: PASS


In [14]:
# =====================================================================
# NOTEBOOK 16 — SECTION 8: EMPIRICAL DISCLOSURE RISK (FROZEN)
# =====================================================================

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------
# 1. Validate source coverage and uniqueness
# ---------------------------------------------------------------------

required_sources = {
    "exact": EXACT_DUPLICATE_DF,
    "near": NEAR_DUPLICATE_DF,
    "distance": DISTANCE_RISK_DF,
}

for source_name, source_df in required_sources.items():
    if "dataset" not in source_df.columns:
        raise ValueError(
            f"{source_name} results do not contain a dataset column."
        )

    if source_df["dataset"].duplicated().any():
        raise ValueError(
            f"{source_name} results contain duplicate dataset entries."
        )

expected_datasets = set(DATASETS)

for source_name, source_df in required_sources.items():
    actual_datasets = set(source_df["dataset"])
    if actual_datasets != expected_datasets:
        raise ValueError(
            f"{source_name} dataset coverage mismatch. "
            f"Expected {sorted(expected_datasets)}, "
            f"found {sorted(actual_datasets)}."
        )

# ---------------------------------------------------------------------
# 2. Merge exact-match, near-duplicate, and distance evidence
# ---------------------------------------------------------------------

DISCLOSURE_RISK_DF = (
    EXACT_DUPLICATE_DF[
        [
            "dataset",
            "exact_duplicate_rate",
            "synthetic_rows_matching_real",
            "status",
        ]
    ]
    .rename(columns={"status": "exact_analysis_status"})
    .merge(
        NEAR_DUPLICATE_DF[
            [
                "dataset",
                "near_duplicate_rate_in_sample",
                "similarity_threshold",
                "nearest_similarity_p95",
                "status",
            ]
        ].rename(columns={"status": "near_analysis_status"}),
        on="dataset",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        DISTANCE_RISK_DF[
            [
                "dataset",
                "synthetic_nn_distance_median",
                "real_nn_distance_median",
                "synthetic_below_real_nn_p05_rate",
                "status",
            ]
        ].rename(columns={"status": "distance_analysis_status"}),
        on="dataset",
        how="inner",
        validate="one_to_one",
    )
)

if set(DISCLOSURE_RISK_DF["dataset"]) != expected_datasets:
    raise ValueError(
        "Disclosure-risk merge did not preserve complete dataset coverage."
    )

# ---------------------------------------------------------------------
# 3. Derive empirical indicators without claiming formal privacy
# ---------------------------------------------------------------------

DISCLOSURE_RISK_DF["empirical_status"] = np.where(
    DISCLOSURE_RISK_DF["exact_duplicate_rate"].gt(0),
    "EXACT_MATCHES_OBSERVED",
    "NO_EXACT_MATCHES_OBSERVED",
)

DISCLOSURE_RISK_DF["analysis_status"] = np.where(
    (
        DISCLOSURE_RISK_DF["exact_analysis_status"].eq("PASS")
        & DISCLOSURE_RISK_DF["near_analysis_status"].eq("PASS")
        & DISCLOSURE_RISK_DF["distance_analysis_status"].eq("PASS")
    ),
    "PASS",
    "FAIL",
)

DISCLOSURE_RISK_DF["interpretation_note"] = (
    "Empirical indicators from sampled comparisons. Interpret exact "
    "matches, near-duplicate similarity, and nearest-neighbor distances "
    "jointly. No observed exact matches do not establish privacy; these "
    "metrics are not formal privacy guarantees."
)

# ---------------------------------------------------------------------
# 4. Persist and reload the disclosure-risk summary
# ---------------------------------------------------------------------

disclosure_path = (
    DIRS["disclosure"] / "empirical_disclosure_risk.csv"
)

DISCLOSURE_RISK_DF.to_csv(disclosure_path, index=False)

DISCLOSURE_RISK_RELOADED = pd.read_csv(disclosure_path)

assert len(DISCLOSURE_RISK_RELOADED) == len(DATASETS)
assert set(DISCLOSURE_RISK_RELOADED["dataset"]) == expected_datasets
assert DISCLOSURE_RISK_RELOADED["dataset"].is_unique
assert DISCLOSURE_RISK_RELOADED["analysis_status"].eq("PASS").all()

numeric_columns = [
    "exact_duplicate_rate",
    "near_duplicate_rate_in_sample",
    "similarity_threshold",
    "nearest_similarity_p95",
    "synthetic_nn_distance_median",
    "real_nn_distance_median",
    "synthetic_below_real_nn_p05_rate",
]

for column in numeric_columns:
    if not np.isfinite(
        DISCLOSURE_RISK_RELOADED[column].to_numpy(dtype=float)
    ).all():
        raise ValueError(
            f"Nonfinite values found in disclosure metric: {column}"
        )

display(DISCLOSURE_RISK_DF)

print(f"Disclosure-risk artifact: {disclosure_path}")
print("Empirical disclosure-risk validation: PASS")

,dataset,exact_duplicate_rate,synthetic_rows_matching_real,exact_analysis_status,near_duplicate_rate_in_sample,similarity_threshold,nearest_similarity_p95,near_analysis_status,synthetic_nn_distance_median,real_nn_distance_median,synthetic_below_real_nn_p05_rate,distance_analysis_status,empirical_status,analysis_status,interpretation_note
0,adult_income,0.0,0,PASS,0.3336,0.95,0.983658,PASS,0.084399,0.022561,0.0000,PASS,NO_EXACT_MATCHES_OBSERVED,PASS,Empirical indicators from sampled comparisons....
1,bank_marketing,0.0,0,PASS,0.2612,0.95,0.985996,PASS,0.075786,0.018386,0.0004,PASS,NO_EXACT_MATCHES_OBSERVED,PASS,Empirical indicators from sampled comparisons....
2,diabetes_130us,0.0,0,PASS,0.0000,0.95,0.846390,PASS,0.178375,0.098182,0.0000,PASS,NO_EXACT_MATCHES_OBSERVED,PASS,Empirical indicators from sampled comparisons....


Disclosure-risk artifact: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/disclosure_risk/empirical_disclosure_risk.csv
Empirical disclosure-risk validation: PASS


In [15]:
# =====================================================================
# SECTION 9 — ADDITIONAL PRIVACY ATTACK ANALYSIS, IF DEFINED
# =====================================================================

def find_attack_registry(config):
    candidate_keys = [
        "privacy_attack_registry",
        "privacy_attacks",
        "attack_registry",
    ]

    for key in candidate_keys:
        value = config.get(key)
        if value is not None:
            return key, value

    # Check a nested privacy section if the configuration uses one.
    privacy_section = config.get("privacy", {})
    if isinstance(privacy_section, dict):
        for key in candidate_keys:
            value = privacy_section.get(key)
            if value is not None:
                return f"privacy.{key}", value

    return None, None

ATTACK_REGISTRY_KEY, ATTACK_REGISTRY = find_attack_registry(MASTER_CONFIG)

ATTACK_RESULTS = []

if ATTACK_REGISTRY is None:
    ATTACK_RESULTS.append({
        "attack_name": None,
        "status": "SKIPPED_NOT_CONFIGURED",
        "reason": "No explicit privacy attack registry found in master_config.yaml.",
        "metrics": None,
    })
else:
    # This notebook intentionally does not guess attack semantics from names.
    if isinstance(ATTACK_REGISTRY, dict):
        attack_items = [
            {"attack_name": name, **(spec if isinstance(spec, dict) else {})}
            for name, spec in ATTACK_REGISTRY.items()
        ]
    elif isinstance(ATTACK_REGISTRY, list):
        attack_items = [
            item for item in ATTACK_REGISTRY if isinstance(item, dict)
        ]
    else:
        attack_items = []

    enabled_attacks = [
        item for item in attack_items
        if item.get("enabled", False) is True
    ]

    if not enabled_attacks:
        ATTACK_RESULTS.append({
            "attack_name": None,
            "status": "SKIPPED_NOT_ENABLED",
            "reason": f"Registry found at {ATTACK_REGISTRY_KEY}, but no attack is explicitly enabled.",
            "metrics": None,
        })
    else:
        for attack in enabled_attacks:
            ATTACK_RESULTS.append({
                "attack_name": attack.get("attack_name", "UNNAMED_ATTACK"),
                "status": "NOT_EXECUTED_UNSUPPORTED_PROTOCOL",
                "reason": (
                    "Attack is enabled in configuration, but Notebook 16 requires "
                    "an explicit implementation, threat model, membership labels, "
                    "and train/holdout protocol before execution."
                ),
                "metrics": None,
            })

ATTACK_RESULTS_DF = pd.DataFrame(ATTACK_RESULTS)
ATTACK_RESULTS_DF.to_csv(
    DIRS["attacks"] / "privacy_attack_results.csv",
    index=False,
)

display(ATTACK_RESULTS_DF)

,attack_name,status,reason,metrics
0,None,SKIPPED_NOT_CONFIGURED,No explicit privacy attack registry found in m...,None


In [17]:
# =====================================================================
# NOTEBOOK 16 — SECTION 10: DATASET-LEVEL PRIVACY SUMMARY (FROZEN)
# =====================================================================

import numpy as np
import pandas as pd

DATASET_PRIVACY_SUMMARY_ROWS = []

# ---------------------------------------------------------------------
# 1. Validate source coverage
# ---------------------------------------------------------------------

EXPECTED_DATASETS = set(DATASETS)

SUMMARY_SOURCES = {
    "exact": EXACT_DUPLICATE_DF,
    "near": NEAR_DUPLICATE_DF,
    "distance": DISTANCE_RISK_DF,
    "disclosure": DISCLOSURE_RISK_DF,
}

for source_name, source_df in SUMMARY_SOURCES.items():
    if source_df["dataset"].duplicated().any():
        raise ValueError(
            f"Duplicate dataset entries in {source_name} results."
        )

    if set(source_df["dataset"]) != EXPECTED_DATASETS:
        raise ValueError(
            f"Dataset coverage mismatch in {source_name} results."
        )

# ---------------------------------------------------------------------
# 2. Construct dataset-level summary
# ---------------------------------------------------------------------

for dataset_name in DATASETS:
    real_df = ALIGNED_REAL[dataset_name]
    synth_df = ALIGNED_SYNTHETIC[dataset_name]

    exact = EXACT_DUPLICATE_DF.loc[
        EXACT_DUPLICATE_DF["dataset"] == dataset_name
    ].iloc[0]

    near = NEAR_DUPLICATE_DF.loc[
        NEAR_DUPLICATE_DF["dataset"] == dataset_name
    ].iloc[0]

    distance = DISTANCE_RISK_DF.loc[
        DISTANCE_RISK_DF["dataset"] == dataset_name
    ].iloc[0]

    disclosure = DISCLOSURE_RISK_DF.loc[
        DISCLOSURE_RISK_DF["dataset"] == dataset_name
    ].iloc[0]

    DATASET_PRIVACY_SUMMARY_ROWS.append({
        "dataset": dataset_name,
        "real_rows": len(real_df),
        "synthetic_rows": len(synth_df),
        "feature_count": len(real_df.columns),
        "exact_duplicate_count": int(
            exact["synthetic_rows_matching_real"]
        ),
        "exact_duplicate_rate": float(
            exact["exact_duplicate_rate"]
        ),
        "near_duplicate_sample_rows": int(
            near["near_duplicate_count_in_sample"]
        ),
        "near_duplicate_sample_size": int(
            near["synthetic_rows_sampled"]
        ),
        "near_duplicate_sample_rate": float(
            near["near_duplicate_rate_in_sample"]
        ),
        "near_duplicate_threshold": float(
            near["similarity_threshold"]
        ),
        "nearest_similarity_p95": float(
            near["nearest_similarity_p95"]
        ),
        "synthetic_nn_distance_median": float(
            distance["synthetic_nn_distance_median"]
        ),
        "real_nn_distance_median": float(
            distance["real_nn_distance_median"]
        ),
        "synthetic_below_real_nn_p05_rate": float(
            distance["synthetic_below_real_nn_p05_rate"]
        ),
        "empirical_status": disclosure["empirical_status"],
        "source_analysis_status": (
            "PASS"
            if (
                exact["status"] == "PASS"
                and near["status"] == "PASS"
                and distance["status"] == "PASS"
                and disclosure["analysis_status"] == "PASS"
            )
            else "FAIL"
        ),
        "interpretation": (
            "Empirical disclosure indicators only. Exact-match absence "
            "does not establish privacy. Near-duplicate and distance "
            "metrics are sample-dependent indicators, not formal privacy "
            "guarantees."
        ),
    })

DATASET_PRIVACY_SUMMARY_DF = pd.DataFrame(
    DATASET_PRIVACY_SUMMARY_ROWS
)

# ---------------------------------------------------------------------
# 3. Validate summary values and persist artifact
# ---------------------------------------------------------------------

if len(DATASET_PRIVACY_SUMMARY_DF) != len(DATASETS):
    raise ValueError("Dataset-level summary row count mismatch.")

if set(DATASET_PRIVACY_SUMMARY_DF["dataset"]) != EXPECTED_DATASETS:
    raise ValueError("Dataset-level summary coverage mismatch.")

if DATASET_PRIVACY_SUMMARY_DF["dataset"].duplicated().any():
    raise ValueError("Dataset-level summary contains duplicate datasets.")

if not DATASET_PRIVACY_SUMMARY_DF["source_analysis_status"].eq("PASS").all():
    raise ValueError("One or more source analyses did not pass validation.")

RATE_COLUMNS = [
    "exact_duplicate_rate",
    "near_duplicate_sample_rate",
    "synthetic_below_real_nn_p05_rate",
]

for column in RATE_COLUMNS:
    values = DATASET_PRIVACY_SUMMARY_DF[column].to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise ValueError(f"Nonfinite values found in {column}.")

    if not ((values >= 0.0) & (values <= 1.0)).all():
        raise ValueError(f"Out-of-range rates found in {column}.")

summary_path = DIRS["summary"] / "dataset_privacy_summary.csv"
DATASET_PRIVACY_SUMMARY_DF.to_csv(summary_path, index=False)

# Reload to verify persistence.
DATASET_PRIVACY_SUMMARY_RELOADED = pd.read_csv(summary_path)

assert len(DATASET_PRIVACY_SUMMARY_RELOADED) == len(DATASETS)
assert set(DATASET_PRIVACY_SUMMARY_RELOADED["dataset"]) == EXPECTED_DATASETS
assert DATASET_PRIVACY_SUMMARY_RELOADED["dataset"].is_unique
assert DATASET_PRIVACY_SUMMARY_RELOADED[
    "source_analysis_status"
].eq("PASS").all()

display(DATASET_PRIVACY_SUMMARY_DF)

print(f"Dataset privacy summary saved: {summary_path}")
print("Dataset-level privacy summary validation: PASS")

,dataset,real_rows,synthetic_rows,feature_count,exact_duplicate_count,exact_duplicate_rate,near_duplicate_sample_rows,near_duplicate_sample_size,near_duplicate_sample_rate,near_duplicate_threshold,nearest_similarity_p95,synthetic_nn_distance_median,real_nn_distance_median,synthetic_below_real_nn_p05_rate,empirical_status,source_analysis_status,interpretation
0,adult_income,48842,34189,15,0,0.0,1668,5000,0.3336,0.95,0.983658,0.084399,0.022561,0.0000,NO_EXACT_MATCHES_OBSERVED,PASS,Empirical disclosure indicators only. Exact-ma...
1,bank_marketing,45211,31647,17,0,0.0,1306,5000,0.2612,0.95,0.985996,0.075786,0.018386,0.0004,NO_EXACT_MATCHES_OBSERVED,PASS,Empirical disclosure indicators only. Exact-ma...
2,diabetes_130us,101766,71236,48,0,0.0,0,5000,0.0000,0.95,0.846390,0.178375,0.098182,0.0000,NO_EXACT_MATCHES_OBSERVED,PASS,Empirical disclosure indicators only. Exact-ma...


Dataset privacy summary saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/summaries/dataset_privacy_summary.csv
Dataset-level privacy summary validation: PASS


In [19]:
# =====================================================================
# NOTEBOOK 16 — SECTION 11: MODEL-LEVEL PRIVACY SUMMARY (FROZEN)
# =====================================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ---------------------------------------------------------------------
# 1. Explicitly register synthetic generator provenance
# ---------------------------------------------------------------------
# Do not infer the generating model from the artifact's directory name.
# Notebook 13 output filenames alone do not establish model provenance.

MODEL_PROVENANCE = {
    "adult_income": {
        "model": "SPP-GAN",
        "provenance_status": "UNVERIFIED",
    },
    "bank_marketing": {
        "model": "SPP-GAN",
        "provenance_status": "UNVERIFIED",
    },
    "diabetes_130us": {
        "model": "SPP-GAN",
        "provenance_status": "UNVERIFIED",
    },
}

if set(MODEL_PROVENANCE) != set(DATASETS):
    raise ValueError("Model provenance registry does not cover all datasets.")

# ---------------------------------------------------------------------
# 2. Validate dataset-level summary coverage
# ---------------------------------------------------------------------

if DATASET_PRIVACY_SUMMARY_DF["dataset"].duplicated().any():
    raise ValueError("Dataset privacy summary contains duplicate datasets.")

if set(DATASET_PRIVACY_SUMMARY_DF["dataset"]) != set(DATASETS):
    raise ValueError("Dataset privacy summary coverage mismatch.")

MODEL_LEVEL_ROWS = []

for dataset_name in DATASETS:
    summary_matches = DATASET_PRIVACY_SUMMARY_DF.loc[
        DATASET_PRIVACY_SUMMARY_DF["dataset"] == dataset_name
    ]

    if len(summary_matches) != 1:
        raise ValueError(
            f"Expected exactly one dataset summary row for {dataset_name}."
        )

    row = summary_matches.iloc[0].to_dict()
    provenance = MODEL_PROVENANCE[dataset_name]
    artifact_path = SYNTHETIC_DATA_PATHS[dataset_name]

    row["model"] = provenance["model"]
    row["provenance_status"] = provenance["provenance_status"]
    row["synthetic_artifact_path"] = str(artifact_path)

    # Confirm that the recorded artifact path is the one configured
    # and loaded in Section 3.
    if not artifact_path.is_file():
        raise FileNotFoundError(
            f"Synthetic artifact missing for {dataset_name}: {artifact_path}"
        )

    row["artifact_exists"] = True
    MODEL_LEVEL_ROWS.append(row)

MODEL_LEVEL_PRIVACY_DF = pd.DataFrame(MODEL_LEVEL_ROWS)

# ---------------------------------------------------------------------
# 3. Save and reload dataset/model-level results
# ---------------------------------------------------------------------

model_level_path = (
    DIRS["summary"] / "model_level_privacy_summary.csv"
)

MODEL_LEVEL_PRIVACY_DF.to_csv(model_level_path, index=False)

MODEL_LEVEL_RELOADED = pd.read_csv(model_level_path)

assert len(MODEL_LEVEL_RELOADED) == len(DATASETS)
assert set(MODEL_LEVEL_RELOADED["dataset"]) == set(DATASETS)
assert MODEL_LEVEL_RELOADED["dataset"].is_unique
assert MODEL_LEVEL_RELOADED["artifact_exists"].all()

# ---------------------------------------------------------------------
# 4. Aggregate descriptive metrics by model
# ---------------------------------------------------------------------
# Aggregation is permitted only when model provenance is verified.
# Unverified provenance must not be presented as a confirmed model-level
# privacy result.

MODEL_NUMERIC_COLUMNS = [
    "exact_duplicate_rate",
    "near_duplicate_sample_rate",
    "nearest_similarity_p95",
    "synthetic_nn_distance_median",
    "real_nn_distance_median",
    "synthetic_below_real_nn_p05_rate",
]

verified_rows = MODEL_LEVEL_PRIVACY_DF.loc[
    MODEL_LEVEL_PRIVACY_DF["provenance_status"] == "VERIFIED"
].copy()

if verified_rows.empty:
    MODEL_AGGREGATION = pd.DataFrame(columns=[
        "model",
        "dataset_count",
        *MODEL_NUMERIC_COLUMNS,
        "aggregation_note",
    ])
else:
    MODEL_AGGREGATION = (
        verified_rows
        .groupby("model", dropna=False)
        .agg(
            dataset_count=("dataset", "nunique"),
            **{
                column: (column, "mean")
                for column in MODEL_NUMERIC_COLUMNS
            },
        )
        .reset_index()
    )

    MODEL_AGGREGATION["aggregation_note"] = (
        "Unweighted mean across datasets with verified model provenance; "
        "descriptive only, not a formal privacy guarantee."
    )

aggregation_path = (
    DIRS["summary"] / "model_privacy_aggregation.csv"
)

MODEL_AGGREGATION.to_csv(aggregation_path, index=False)

MODEL_AGGREGATION_RELOADED = pd.read_csv(aggregation_path)

assert len(MODEL_AGGREGATION_RELOADED) == len(MODEL_AGGREGATION)

display(MODEL_LEVEL_PRIVACY_DF)
display(MODEL_AGGREGATION)

print(f"Model-level summary saved: {model_level_path}")
print(f"Model aggregation saved: {aggregation_path}")
print("Model-level artifact validation: PASS")
print(
    "Model aggregation status:",
    "PASS" if not MODEL_AGGREGATION.empty
    else "SKIPPED — generator provenance is unverified"
)

,dataset,real_rows,synthetic_rows,feature_count,exact_duplicate_count,exact_duplicate_rate,near_duplicate_sample_rows,near_duplicate_sample_size,near_duplicate_sample_rate,near_duplicate_threshold,...,synthetic_nn_distance_median,real_nn_distance_median,synthetic_below_real_nn_p05_rate,empirical_status,source_analysis_status,interpretation,model,provenance_status,synthetic_artifact_path,artifact_exists
0,adult_income,48842,34189,15,0,0.0,1668,5000,0.3336,0.95,...,0.084399,0.022561,0.0000,NO_EXACT_MATCHES_OBSERVED,PASS,Empirical disclosure indicators only. Exact-ma...,SPP-GAN,UNVERIFIED,/content/drive/MyDrive/SPP_GAN_Research/result...,True
1,bank_marketing,45211,31647,17,0,0.0,1306,5000,0.2612,0.95,...,0.075786,0.018386,0.0004,NO_EXACT_MATCHES_OBSERVED,PASS,Empirical disclosure indicators only. Exact-ma...,SPP-GAN,UNVERIFIED,/content/drive/MyDrive/SPP_GAN_Research/result...,True
2,diabetes_130us,101766,71236,48,0,0.0,0,5000,0.0000,0.95,...,0.178375,0.098182,0.0000,NO_EXACT_MATCHES_OBSERVED,PASS,Empirical disclosure indicators only. Exact-ma...,SPP-GAN,UNVERIFIED,/content/drive/MyDrive/SPP_GAN_Research/result...,True


,model,dataset_count,exact_duplicate_rate,near_duplicate_sample_rate,nearest_similarity_p95,synthetic_nn_distance_median,real_nn_distance_median,synthetic_below_real_nn_p05_rate,aggregation_note


Model-level summary saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/summaries/model_level_privacy_summary.csv
Model aggregation saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/summaries/model_privacy_aggregation.csv
Model-level artifact validation: PASS
Model aggregation status: SKIPPED — generator provenance is unverified


In [21]:
# =====================================================================
# NOTEBOOK 16 — SECTION 12: SAVE PRIVACY-RISK RESULTS (FROZEN)
# =====================================================================

import hashlib
import json
import pandas as pd
from datetime import datetime, timezone

# ---------------------------------------------------------------------
# 1. File hashing utility
# ---------------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)

    return digest.hexdigest()


# ---------------------------------------------------------------------
# 2. Validate source coverage before consolidation
# ---------------------------------------------------------------------

EXPECTED_DATASETS = set(DATASETS)

if set(DATASET_PRIVACY_SUMMARY_DF["dataset"]) != EXPECTED_DATASETS:
    raise ValueError("Dataset privacy summary coverage mismatch.")

if set(ALIGNMENT_DF["dataset"]) != EXPECTED_DATASETS:
    raise ValueError("Alignment validation coverage mismatch.")

if DATASET_PRIVACY_SUMMARY_DF["dataset"].duplicated().any():
    raise ValueError("Duplicate datasets in privacy summary.")

if ALIGNMENT_DF["dataset"].duplicated().any():
    raise ValueError("Duplicate datasets in alignment results.")

# ---------------------------------------------------------------------
# 3. Consolidate privacy-risk results
# ---------------------------------------------------------------------

PRIVACY_RESULTS_DF = DATASET_PRIVACY_SUMMARY_DF.merge(
    ALIGNMENT_DF[
        ["dataset", "schema_status", "real_columns", "synthetic_columns"]
    ],
    on="dataset",
    how="inner",
    validate="one_to_one",
)

# Preserve the explicitly configured generator provenance status.
PRIVACY_RESULTS_DF["generator_label"] = PRIVACY_RESULTS_DF["dataset"].map(
    {
        dataset: MODEL_PROVENANCE[dataset]["model"]
        for dataset in DATASETS
    }
)

PRIVACY_RESULTS_DF["generator_provenance_status"] = (
    PRIVACY_RESULTS_DF["dataset"].map(
        {
            dataset: MODEL_PROVENANCE[dataset]["provenance_status"]
            for dataset in DATASETS
        }
    )
)

PRIVACY_RESULTS_DF["real_data_path"] = PRIVACY_RESULTS_DF["dataset"].map(
    {dataset: str(path) for dataset, path in REAL_DATA_PATHS.items()}
)

PRIVACY_RESULTS_DF["synthetic_data_path"] = (
    PRIVACY_RESULTS_DF["dataset"].map(
        {dataset: str(path) for dataset, path in SYNTHETIC_DATA_PATHS.items()}
    )
)

PRIVACY_RESULTS_DF["evaluation_timestamp_utc"] = RUN_STARTED_UTC
PRIVACY_RESULTS_DF["privacy_seed"] = PRIVACY_SEED
PRIVACY_RESULTS_DF["max_real_reference_rows"] = MAX_REAL_REFERENCE_ROWS
PRIVACY_RESULTS_DF["max_synthetic_rows"] = MAX_SYNTHETIC_ROWS
PRIVACY_RESULTS_DF["near_duplicate_threshold"] = NEAR_DUPLICATE_THRESHOLD
PRIVACY_RESULTS_DF["synthetic_generation_repetitions"] = 1

if len(PRIVACY_RESULTS_DF) != len(DATASETS):
    raise ValueError("Consolidated results have an unexpected row count.")

if set(PRIVACY_RESULTS_DF["dataset"]) != EXPECTED_DATASETS:
    raise ValueError("Consolidated results have incomplete dataset coverage.")

if not PRIVACY_RESULTS_DF["schema_status"].eq("PASS").all():
    raise ValueError("Schema alignment did not pass for every dataset.")

if not PRIVACY_RESULTS_DF["source_analysis_status"].eq("PASS").all():
    raise ValueError("One or more source privacy analyses did not pass.")

# ---------------------------------------------------------------------
# 4. Save consolidated privacy results
# ---------------------------------------------------------------------

PRIVACY_RESULTS_PATH = DIRS["results"] / "privacy_risk_results.csv"
PRIVACY_RESULTS_DF.to_csv(PRIVACY_RESULTS_PATH, index=False)

# ---------------------------------------------------------------------
# 5. Register key output artifacts
# ---------------------------------------------------------------------

OUTPUT_TABLES = {
    "real_data_discovery": DIRS["inputs"] / "real_data_discovery.csv",
    "synthetic_data_discovery": DIRS["inputs"] / "synthetic_data_discovery.csv",
    "synthetic_data_fingerprints": DIRS["inputs"] / "synthetic_data_fingerprints.csv",
    "alignment_validation": DIRS["inputs"] / "alignment_validation.csv",
    "privacy_input_manifest": DIRS["inputs"] / "privacy_input_manifest.csv",
    "exact_duplicate_summary": DIRS["exact"] / "exact_duplicate_summary.csv",
    "exact_duplicate_details": DIRS["exact"] / "exact_duplicate_details.csv",
    "near_duplicate_summary": DIRS["near"] / "near_duplicate_summary.csv",
    "near_duplicate_score_files": DIRS["near"] / "near_duplicate_score_files.csv",
    "nearest_neighbor_risk_summary": DIRS["distance"] / "nearest_neighbor_risk_summary.csv",
    "nearest_neighbor_distance_files": DIRS["distance"] / "nearest_neighbor_distance_files.csv",
    "empirical_disclosure_risk": DIRS["disclosure"] / "empirical_disclosure_risk.csv",
    "privacy_attack_results": DIRS["attacks"] / "privacy_attack_results.csv",
    "dataset_privacy_summary": DIRS["summary"] / "dataset_privacy_summary.csv",
    "model_level_privacy_summary": DIRS["summary"] / "model_level_privacy_summary.csv",
    "model_privacy_aggregation": DIRS["summary"] / "model_privacy_aggregation.csv",
    "privacy_risk_results": PRIVACY_RESULTS_PATH,
}

# ---------------------------------------------------------------------
# 6. Create input provenance manifest
# ---------------------------------------------------------------------

INPUT_MANIFEST_ROWS = []

for dataset_name in DATASETS:
    for input_type, path in [
        ("real", REAL_DATA_PATHS[dataset_name]),
        ("synthetic", SYNTHETIC_DATA_PATHS[dataset_name]),
    ]:
        if not path.is_file():
            raise FileNotFoundError(
                f"Input artifact missing: {path}"
            )

        INPUT_MANIFEST_ROWS.append({
            "dataset": dataset_name,
            "input_type": input_type,
            "path": str(path),
            "exists": True,
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })

INPUT_MANIFEST_DF = pd.DataFrame(INPUT_MANIFEST_ROWS)

INPUT_MANIFEST_PATH = DIRS["inputs"] / "privacy_input_manifest.csv"
INPUT_MANIFEST_DF.to_csv(INPUT_MANIFEST_PATH, index=False)

# ---------------------------------------------------------------------
# 7. Fingerprint saved outputs
# ---------------------------------------------------------------------

FINGERPRINT_ROWS = []

for artifact_name, path in OUTPUT_TABLES.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Expected output artifact is missing: {path}"
        )

    FINGERPRINT_ROWS.append({
        "artifact_name": artifact_name,
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })

FINGERPRINT_DF = pd.DataFrame(FINGERPRINT_ROWS)

FINGERPRINT_PATH = (
    DIRS["validation"] / "privacy_artifact_fingerprints.csv"
)

FINGERPRINT_DF.to_csv(FINGERPRINT_PATH, index=False)

# ---------------------------------------------------------------------
# 8. Reload and verify persisted artifacts
# ---------------------------------------------------------------------

PRIVACY_RESULTS_RELOADED = pd.read_csv(PRIVACY_RESULTS_PATH)
INPUT_MANIFEST_RELOADED = pd.read_csv(INPUT_MANIFEST_PATH)
FINGERPRINTS_RELOADED = pd.read_csv(FINGERPRINT_PATH)

assert len(PRIVACY_RESULTS_RELOADED) == len(DATASETS)
assert set(PRIVACY_RESULTS_RELOADED["dataset"]) == EXPECTED_DATASETS
assert len(INPUT_MANIFEST_RELOADED) == len(DATASETS) * 2
assert len(FINGERPRINTS_RELOADED) == len(OUTPUT_TABLES)

assert INPUT_MANIFEST_RELOADED["exists"].all()
assert INPUT_MANIFEST_RELOADED["sha256"].notna().all()
assert FINGERPRINTS_RELOADED["sha256"].notna().all()

# Confirm that every fingerprint still matches its saved artifact.
for record in FINGERPRINTS_RELOADED.to_dict("records"):
    artifact_path = Path(record["path"])

    assert artifact_path.is_file(), (
        f"Fingerprint artifact is missing: {artifact_path}"
    )
    assert sha256_file(artifact_path) == record["sha256"], (
        f"Fingerprint mismatch: {artifact_path}"
    )

print("Privacy results saved:", PRIVACY_RESULTS_PATH)
print("Input manifest saved:", INPUT_MANIFEST_PATH)
print("Output fingerprint manifest saved:", FINGERPRINT_PATH)
print("Consolidated rows:", len(PRIVACY_RESULTS_RELOADED))

display(PRIVACY_RESULTS_RELOADED)

print("Section 12 artifact validation: PASS")

Privacy results saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/results/privacy_risk_results.csv
Input manifest saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/inputs/privacy_input_manifest.csv
Output fingerprint manifest saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/validation/privacy_artifact_fingerprints.csv
Consolidated rows: 3


,dataset,real_rows,synthetic_rows,feature_count,exact_duplicate_count,exact_duplicate_rate,near_duplicate_sample_rows,near_duplicate_sample_size,near_duplicate_sample_rate,near_duplicate_threshold,...,synthetic_columns,generator_label,generator_provenance_status,real_data_path,synthetic_data_path,evaluation_timestamp_utc,privacy_seed,max_real_reference_rows,max_synthetic_rows,synthetic_generation_repetitions
0,adult_income,48842,34189,15,0,0.0,1668,5000,0.3336,0.95,...,15,SPP-GAN,UNVERIFIED,/content/drive/MyDrive/SPP_GAN_Research/data/r...,/content/drive/MyDrive/SPP_GAN_Research/result...,2026-10-26T06:51:08.528764+00:00,1616,5000,5000,1
1,bank_marketing,45211,31647,17,0,0.0,1306,5000,0.2612,0.95,...,17,SPP-GAN,UNVERIFIED,/content/drive/MyDrive/SPP_GAN_Research/data/r...,/content/drive/MyDrive/SPP_GAN_Research/result...,2026-10-26T06:51:08.528764+00:00,1616,5000,5000,1
2,diabetes_130us,101766,71236,48,0,0.0,0,5000,0.0000,0.95,...,48,SPP-GAN,UNVERIFIED,/content/drive/MyDrive/SPP_GAN_Research/data/r...,/content/drive/MyDrive/SPP_GAN_Research/result...,2026-10-26T06:51:08.528764+00:00,1616,5000,5000,1


Section 12 artifact validation: PASS


In [23]:
# =====================================================================
# NOTEBOOK 16 — SECTION 13: VALIDATE COVERAGE (FROZEN)
# =====================================================================

import numpy as np
import pandas as pd

VALIDATION_ROWS = []


def add_check(check_name, passed, observed, expected, details=""):
    VALIDATION_ROWS.append({
        "check": check_name,
        "status": "PASS" if bool(passed) else "FAIL",
        "observed": str(observed),
        "expected": str(expected),
        "details": str(details),
    })


EXPECTED_DATASETS = set(DATASETS)

# ---------------------------------------------------------------------
# 1. Dataset coverage and uniqueness
# ---------------------------------------------------------------------

actual_datasets = set(PRIVACY_RESULTS_DF["dataset"])

add_check(
    "dataset_coverage",
    actual_datasets == EXPECTED_DATASETS
    and PRIVACY_RESULTS_DF["dataset"].is_unique,
    sorted(actual_datasets),
    sorted(EXPECTED_DATASETS),
    "Dataset coverage must be complete and unique.",
)

# ---------------------------------------------------------------------
# 2. Schema alignment
# ---------------------------------------------------------------------

alignment_pass = (
    set(ALIGNMENT_DF["dataset"]) == EXPECTED_DATASETS
    and ALIGNMENT_DF["dataset"].is_unique
    and ALIGNMENT_DF["schema_status"].eq("PASS").all()
)

add_check(
    "schema_alignment",
    alignment_pass,
    int(ALIGNMENT_DF["schema_status"].eq("PASS").sum()),
    len(EXPECTED_DATASETS),
    "Explicitly documented schema exclusions are recorded in Section 4.",
)

# ---------------------------------------------------------------------
# 3. Dataset-level and model-level summary coverage
# ---------------------------------------------------------------------

for check_name, frame in [
    ("dataset_summary_coverage", DATASET_PRIVACY_SUMMARY_DF),
    ("model_summary_coverage", MODEL_LEVEL_PRIVACY_DF),
]:
    passed = (
        set(frame["dataset"]) == EXPECTED_DATASETS
        and frame["dataset"].is_unique
        and len(frame) == len(EXPECTED_DATASETS)
    )

    add_check(
        check_name,
        passed,
        len(frame),
        len(EXPECTED_DATASETS),
        "Each expected dataset must appear exactly once.",
    )

# ---------------------------------------------------------------------
# 4. Metric ranges and finite values
# ---------------------------------------------------------------------

RATE_COLUMNS = [
    "exact_duplicate_rate",
    "near_duplicate_sample_rate",
    "synthetic_below_real_nn_p05_rate",
]

for column in RATE_COLUMNS:
    values = pd.to_numeric(
        PRIVACY_RESULTS_DF[column], errors="coerce"
    )

    valid = (
        values.notna()
        & np.isfinite(values)
        & values.between(0.0, 1.0)
    )

    add_check(
        f"valid_range_{column}",
        bool(valid.all()),
        int(valid.sum()),
        len(PRIVACY_RESULTS_DF),
        "All values must be finite and in [0, 1].",
    )

# ---------------------------------------------------------------------
# 5. Near-duplicate threshold consistency
# ---------------------------------------------------------------------

threshold_ok = (
    set(NEAR_DUPLICATE_DF["dataset"]) == EXPECTED_DATASETS
    and NEAR_DUPLICATE_DF["dataset"].is_unique
    and np.isclose(
        NEAR_DUPLICATE_DF["similarity_threshold"].to_numpy(dtype=float),
        float(NEAR_DUPLICATE_THRESHOLD),
    ).all()
)

add_check(
    "near_duplicate_threshold_consistency",
    threshold_ok,
    sorted(
        NEAR_DUPLICATE_DF["similarity_threshold"].unique().tolist()
    ),
    NEAR_DUPLICATE_THRESHOLD,
)

# ---------------------------------------------------------------------
# 6. Input manifest integrity
# ---------------------------------------------------------------------

input_manifest_ok = (
    len(INPUT_MANIFEST_DF) == len(EXPECTED_DATASETS) * 2
    and INPUT_MANIFEST_DF["exists"].eq(True).all()
    and INPUT_MANIFEST_DF["sha256"].notna().all()
    and INPUT_MANIFEST_DF["sha256"].astype(str).str.len().eq(64).all()
    and not INPUT_MANIFEST_DF.duplicated(
        subset=["dataset", "input_type"]
    ).any()
)

add_check(
    "input_manifest_complete",
    input_manifest_ok,
    len(INPUT_MANIFEST_DF),
    len(EXPECTED_DATASETS) * 2,
    "Every real and synthetic input must have a SHA-256 fingerprint.",
)

# ---------------------------------------------------------------------
# 7. Required artifacts and output fingerprints
# ---------------------------------------------------------------------

required_paths = [
    PRIVACY_RESULTS_PATH,
    INPUT_MANIFEST_PATH,
    DIRS["exact"] / "exact_duplicate_summary.csv",
    DIRS["near"] / "near_duplicate_summary.csv",
    DIRS["distance"] / "nearest_neighbor_risk_summary.csv",
    DIRS["disclosure"] / "empirical_disclosure_risk.csv",
    DIRS["attacks"] / "privacy_attack_results.csv",
    DIRS["summary"] / "dataset_privacy_summary.csv",
    DIRS["summary"] / "model_level_privacy_summary.csv",
    FINGERPRINT_PATH,
]

missing_outputs = [
    str(path)
    for path in required_paths
    if not path.is_file() or path.stat().st_size == 0
]

add_check(
    "required_artifacts_exist",
    len(missing_outputs) == 0,
    len(required_paths) - len(missing_outputs),
    len(required_paths),
    "Missing or empty artifacts: " + "; ".join(missing_outputs),
)

# Verify each registered output fingerprint against the actual file.
fingerprint_errors = []

for record in FINGERPRINTS_RELOADED.to_dict("records"):
    path = Path(record["path"])

    if not path.is_file():
        fingerprint_errors.append(f"Missing: {path}")
    elif sha256_file(path) != record["sha256"]:
        fingerprint_errors.append(f"Hash mismatch: {path}")

add_check(
    "output_fingerprint_integrity",
    len(fingerprint_errors) == 0,
    len(FINGERPRINTS_RELOADED) - len(fingerprint_errors),
    len(FINGERPRINTS_RELOADED),
    "; ".join(fingerprint_errors),
)

# ---------------------------------------------------------------------
# 8. Consolidated results consistency
# ---------------------------------------------------------------------

consolidated_ok = (
    len(PRIVACY_RESULTS_DF) == len(EXPECTED_DATASETS)
    and PRIVACY_RESULTS_DF["dataset"].is_unique
    and PRIVACY_RESULTS_DF["source_analysis_status"].eq("PASS").all()
    and PRIVACY_RESULTS_DF["schema_status"].eq("PASS").all()
)

add_check(
    "consolidated_results_consistency",
    consolidated_ok,
    int(
        (
            PRIVACY_RESULTS_DF["source_analysis_status"].eq("PASS")
            & PRIVACY_RESULTS_DF["schema_status"].eq("PASS")
        ).sum()
    ),
    len(EXPECTED_DATASETS),
    "Consolidated rows must have passing source analyses and schema checks.",
)

# ---------------------------------------------------------------------
# 9. Save and reload validation results
# ---------------------------------------------------------------------

VALIDATION_DF = pd.DataFrame(VALIDATION_ROWS)

VALIDATION_PATH = (
    DIRS["validation"] / "privacy_coverage_validation.csv"
)

VALIDATION_DF.to_csv(VALIDATION_PATH, index=False)

VALIDATION_RELOADED = pd.read_csv(VALIDATION_PATH)

VALIDATION_PASS = bool(
    len(VALIDATION_RELOADED) > 0
    and VALIDATION_RELOADED["status"].eq("PASS").all()
)

assert len(VALIDATION_RELOADED) == len(VALIDATION_ROWS)

display(VALIDATION_RELOADED)

print("Validation artifact:", VALIDATION_PATH)
print("Validation status:", "PASS" if VALIDATION_PASS else "FAIL")
print(
    "Failed checks:",
    int(VALIDATION_RELOADED["status"].ne("PASS").sum()),
)

if not VALIDATION_PASS:
    print("Inspect failed checks in:", VALIDATION_PATH)

,check,status,observed,expected,details
0,dataset_coverage,PASS,"['adult_income', 'bank_marketing', 'diabetes_1...","['adult_income', 'bank_marketing', 'diabetes_1...",Dataset coverage must be complete and unique.
1,schema_alignment,PASS,3,3,Explicitly documented schema exclusions are re...
2,dataset_summary_coverage,PASS,3,3,Each expected dataset must appear exactly once.
3,model_summary_coverage,PASS,3,3,Each expected dataset must appear exactly once.
4,valid_range_exact_duplicate_rate,PASS,3,3,"All values must be finite and in [0, 1]."
5,valid_range_near_duplicate_sample_rate,PASS,3,3,"All values must be finite and in [0, 1]."
6,valid_range_synthetic_below_real_nn_p05_rate,PASS,3,3,"All values must be finite and in [0, 1]."
7,near_duplicate_threshold_consistency,PASS,[0.95],0.95,NaN
8,input_manifest_complete,PASS,6,6,Every real and synthetic input must have a SHA...
9,required_artifacts_exist,PASS,10,10,Missing or empty artifacts:


Validation artifact: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/validation/privacy_coverage_validation.csv
Validation status: PASS
Failed checks: 0


In [25]:
# =====================================================================
# SECTION 14 — COMPLETION SUMMARY
# =====================================================================

from datetime import datetime, timezone
from pathlib import Path
import json
import hashlib

print("=" * 88)
print("SECTION 14 — COMPLETION SUMMARY")
print("=" * 88)

# ---------------------------------------------------------------------
# 1. Validate required runtime objects
# ---------------------------------------------------------------------

required_objects = [
    "VALIDATION_RELOADED",
    "VALIDATION_PASS",
    "RUN_STARTED_UTC",
    "PROJECT_ROOT",
    "NOTEBOOK_ROOT",
    "CONFIG_PATH",
    "DATASETS",
    "PRIVACY_RESULTS_DF",
    "MAX_REAL_REFERENCE_ROWS",
    "MAX_SYNTHETIC_ROWS",
    "PAIRWISE_CHUNK_SIZE",
    "NEAR_DUPLICATE_THRESHOLD",
    "PRIVACY_RESULTS_PATH",
    "DIRS",
    "INPUT_MANIFEST_PATH",
    "FINGERPRINT_PATH",
    "VALIDATION_PATH",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 14 cannot run. Required objects are missing: "
        + ", ".join(missing_objects)
    )

if not isinstance(VALIDATION_RELOADED, pd.DataFrame):
    raise TypeError("VALIDATION_RELOADED must be a pandas DataFrame.")

if not isinstance(PRIVACY_RESULTS_DF, pd.DataFrame):
    raise TypeError("PRIVACY_RESULTS_DF must be a pandas DataFrame.")

if "dataset" not in PRIVACY_RESULTS_DF.columns:
    raise ValueError(
        "PRIVACY_RESULTS_DF must contain a 'dataset' column."
    )

if "status" not in VALIDATION_RELOADED.columns:
    raise ValueError(
        "VALIDATION_RELOADED must contain a 'status' column."
    )

# ---------------------------------------------------------------------
# 2. Determine final validation status
# ---------------------------------------------------------------------

failed_checks = int(
    (VALIDATION_RELOADED["status"].astype(str) != "PASS").sum()
)

validation_rows_present = len(VALIDATION_RELOADED) > 0

validation_pass_consistent = (
    bool(VALIDATION_PASS)
    and validation_rows_present
    and failed_checks == 0
)

validation_status = (
    "PASS" if validation_pass_consistent else "FAIL"
)

completion_status = (
    "COMPLETED" if validation_pass_consistent else "PARTIAL"
)

# ---------------------------------------------------------------------
# 3. Validate dataset coverage
# ---------------------------------------------------------------------

expected_datasets = list(DATASETS)
observed_datasets = PRIVACY_RESULTS_DF["dataset"].astype(str).tolist()

expected_dataset_set = set(expected_datasets)
observed_dataset_set = set(observed_datasets)

dataset_coverage_complete = (
    len(observed_datasets) == len(expected_datasets)
    and len(observed_dataset_set) == len(observed_datasets)
    and observed_dataset_set == expected_dataset_set
)

if not dataset_coverage_complete:
    completion_status = "PARTIAL"
    validation_status = "FAIL"

dataset_count = int(len(observed_dataset_set))

# ---------------------------------------------------------------------
# 4. Capture additional privacy-attack status
# ---------------------------------------------------------------------

if "ATTACK_RESULTS_DF" in globals() and isinstance(
    ATTACK_RESULTS_DF, pd.DataFrame
):
    if (
        not ATTACK_RESULTS_DF.empty
        and "status" in ATTACK_RESULTS_DF.columns
    ):
        attack_statuses = sorted(
            ATTACK_RESULTS_DF["status"]
            .fillna("UNKNOWN")
            .astype(str)
            .unique()
            .tolist()
        )
    else:
        attack_statuses = ["SKIPPED_NOT_CONFIGURED"]
else:
    attack_statuses = ["NOT_REPORTED"]

# ---------------------------------------------------------------------
# 5. Record model provenance status without claiming verification
# ---------------------------------------------------------------------

if (
    "MODEL_PRIVACY_SUMMARY_DF" in globals()
    and isinstance(MODEL_PRIVACY_SUMMARY_DF, pd.DataFrame)
    and "provenance_status" in MODEL_PRIVACY_SUMMARY_DF.columns
):
    provenance_statuses = sorted(
        MODEL_PRIVACY_SUMMARY_DF["provenance_status"]
        .fillna("UNKNOWN")
        .astype(str)
        .unique()
        .tolist()
    )
else:
    provenance_statuses = ["NOT_REPORTED"]

# ---------------------------------------------------------------------
# 6. Define expected artifact locations
# ---------------------------------------------------------------------

artifact_paths = {
    "privacy_risk_results": PRIVACY_RESULTS_PATH,
    "dataset_privacy_summary": (
        DIRS["summary"] / "dataset_privacy_summary.csv"
    ),
    "model_level_privacy_summary": (
        DIRS["summary"] / "model_level_privacy_summary.csv"
    ),
    "exact_duplicate_summary": (
        DIRS["exact"] / "exact_duplicate_summary.csv"
    ),
    "near_duplicate_summary": (
        DIRS["near"] / "near_duplicate_summary.csv"
    ),
    "nearest_neighbor_summary": (
        DIRS["distance"] / "nearest_neighbor_risk_summary.csv"
    ),
    "disclosure_risk": (
        DIRS["disclosure"] / "empirical_disclosure_risk.csv"
    ),
    "privacy_attack_results": (
        DIRS["attacks"] / "privacy_attack_results.csv"
    ),
    "input_manifest": INPUT_MANIFEST_PATH,
    "fingerprint_manifest": FINGERPRINT_PATH,
    "coverage_validation": VALIDATION_PATH,
}

artifact_registry = {}

for artifact_name, artifact_path in artifact_paths.items():
    path = Path(artifact_path)

    artifact_registry[artifact_name] = {
        "path": str(path),
        "exists": bool(path.is_file()),
        "size_bytes": int(path.stat().st_size) if path.is_file() else None,
    }

# ---------------------------------------------------------------------
# 7. Build completion summary
# ---------------------------------------------------------------------

COMPLETION_PATH = (
    Path(NOTEBOOK_ROOT) / "notebook_16_completion_summary.json"
)

COMPLETION_PATH.parent.mkdir(parents=True, exist_ok=True)

COMPLETION_SUMMARY = {
    "notebook_id": "notebook_16",
    "title": "Privacy & Disclosure Risk Evaluation",
    "completion_status": completion_status,
    "validation_status": validation_status,
    "failed_validation_checks": int(failed_checks),
    "validation_check_count": int(len(VALIDATION_RELOADED)),
    "dataset_coverage_complete": bool(dataset_coverage_complete),
    "run_started_utc": str(RUN_STARTED_UTC),
    "summary_created_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "notebook_root": str(NOTEBOOK_ROOT),
    "master_config_path": str(CONFIG_PATH),
    "datasets_expected": expected_datasets,
    "datasets_evaluated": sorted(observed_dataset_set),
    "dataset_count": dataset_count,
    "synthetic_generation_repetitions": 1,
    "max_real_reference_rows": int(MAX_REAL_REFERENCE_ROWS),
    "max_synthetic_rows": int(MAX_SYNTHETIC_ROWS),
    "pairwise_chunk_size": int(PAIRWISE_CHUNK_SIZE),
    "near_duplicate_threshold": float(NEAR_DUPLICATE_THRESHOLD),
    "analyses": [
        "exact_duplicate_analysis",
        "near_duplicate_analysis",
        "nearest_neighbor_distance_risk",
        "empirical_disclosure_risk",
        "dataset_level_summary",
        "model_level_summary",
    ],
    "additional_attack_statuses": attack_statuses,
    "model_provenance_statuses": provenance_statuses,
    "dp_accounting_note": (
        "Empirical disclosure-risk indicators are separate from formal "
        "differential privacy accounting. Refer to the applicable training "
        "protocol and its verified privacy accountant. Do not infer an "
        "end-to-end differential privacy guarantee from this notebook."
    ),
    "sampling_note": (
        "Near-duplicate and nearest-neighbor results use deterministic "
        "samples bounded by configured row limits. Exact row hashing checks "
        "all loaded rows."
    ),
    "limitations": [
        "No observed exact match does not prove absence of disclosure risk.",
        "Near-duplicate similarity is threshold-dependent and empirical.",
        "Nearest-neighbor proximity may reflect naturally dense data regions.",
        "Additional privacy attacks are not executed unless an explicit compatible protocol is configured and implemented.",
        "Model provenance remains unverified unless supported by explicit provenance evidence.",
        "Empirical risk indicators are not formal differential privacy guarantees.",
    ],
    "artifacts": artifact_registry,
}

# ---------------------------------------------------------------------
# 8. Persist JSON safely and verify the written artifact
# ---------------------------------------------------------------------

with open(COMPLETION_PATH, "w", encoding="utf-8") as file:
    json.dump(
        COMPLETION_SUMMARY,
        file,
        indent=2,
        allow_nan=False,
    )

with open(COMPLETION_PATH, "r", encoding="utf-8") as file:
    COMPLETION_RELOADED = json.load(file)

# ---------------------------------------------------------------------
# 9. Verify the persisted completion summary
# ---------------------------------------------------------------------

summary_verified = (
    COMPLETION_RELOADED.get("notebook_id") == "notebook_16"
    and COMPLETION_RELOADED.get("completion_status") == completion_status
    and COMPLETION_RELOADED.get("validation_status") == validation_status
    and COMPLETION_RELOADED.get("failed_validation_checks") == failed_checks
    and COMPLETION_RELOADED.get("dataset_count") == dataset_count
    and COMPLETION_RELOADED.get("datasets_expected") == expected_datasets
    and set(COMPLETION_RELOADED.get("datasets_evaluated", []))
        == expected_dataset_set
    and COMPLETION_PATH.is_file()
    and COMPLETION_PATH.stat().st_size > 0
)

if not summary_verified:
    raise RuntimeError(
        "Notebook 16 completion-summary verification failed."
    )

# ---------------------------------------------------------------------
# 10. Final report and stop on incomplete validation
# ---------------------------------------------------------------------

print("Completion status:", completion_status)
print("Validation status:", validation_status)
print("Failed validation checks:", failed_checks)
print("Validation checks:", len(VALIDATION_RELOADED))
print("Dataset coverage:", "PASS" if dataset_coverage_complete else "FAIL")
print("Datasets evaluated:", dataset_count)
print("Additional attack status:", attack_statuses)
print("Model provenance status:", provenance_statuses)
print("Summary path:", COMPLETION_PATH)
print("Summary reload verification:", "PASS" if summary_verified else "FAIL")
print("=" * 88)

if validation_status != "PASS":
    raise RuntimeError(
        "Notebook 16 validation failed or dataset coverage is incomplete. "
        "Review privacy_coverage_validation.csv before proceeding."
    )

print("NOTEBOOK 16 STATUS: COMPLETED")

SECTION 14 — COMPLETION SUMMARY
Completion status: COMPLETED
Validation status: PASS
Failed validation checks: 0
Validation checks: 12
Dataset coverage: PASS
Datasets evaluated: 3
Additional attack status: ['SKIPPED_NOT_CONFIGURED']
Model provenance status: ['NOT_REPORTED']
Summary path: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/notebook_16_completion_summary.json
Summary reload verification: PASS
NOTEBOOK 16 STATUS: COMPLETED


In [27]:
# =====================================================================
# SECTION 15 — FINAL ARTIFACT, HASH & REPRODUCIBILITY AUDIT
# =====================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pandas as pd
import numpy as np

print("=" * 100)
print("SECTION 15 — FINAL ARTIFACT, HASH & REPRODUCIBILITY AUDIT")
print("=" * 100)

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Research")
NOTEBOOK_ROOT = (
    PROJECT_ROOT / "results" / "notebooks" / "notebook_16"
)
AUDIT_DIR = NOTEBOOK_ROOT / "audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

COMPLETION_PATH = NOTEBOOK_ROOT / "notebook_16_completion_summary.json"
AUDIT_CSV_PATH = AUDIT_DIR / "final_artifact_audit.csv"
AUDIT_JSON_PATH = AUDIT_DIR / "final_artifact_audit_summary.json"

# ---------------------------------------------------------------------
# 1. Utilities
# ---------------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_csv(path):
    return pd.read_csv(path, low_memory=False)


def valid_sha256(value):
    value = str(value).strip().lower()
    return (
        len(value) == 64
        and all(c in "0123456789abcdef" for c in value)
    )


def add_check(check, path, status, observed="", expected="", details=""):
    audit_rows.append({
        "check": check,
        "path": str(path),
        "status": status,
        "observed": str(observed),
        "expected": str(expected),
        "details": str(details),
    })


def resolve_path(raw_path):
    path = Path(str(raw_path)).expanduser()
    if path.is_absolute():
        return path

    candidates = [
        PROJECT_ROOT / path,
        NOTEBOOK_ROOT / path,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    return candidates[0]


def discover_manifest(candidates, patterns):
    """Prefer configured candidates, then search the notebook output tree."""
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.is_file():
            return candidate

    matches = []
    for pattern in patterns:
        matches.extend(
            p for p in NOTEBOOK_ROOT.rglob(pattern)
            if p.is_file()
            and AUDIT_DIR not in p.parents
        )

    matches = sorted(set(matches), key=lambda p: (len(str(p)), str(p)))
    return matches[0] if matches else None


audit_rows = []

# ---------------------------------------------------------------------
# 2. Load completion summary and artifact registry
# ---------------------------------------------------------------------

if not COMPLETION_PATH.is_file():
    raise FileNotFoundError(
        f"Completion summary not found: {COMPLETION_PATH}"
    )

completion = read_json(COMPLETION_PATH)

if completion.get("notebook_id") != "notebook_16":
    raise RuntimeError(
        "Completion JSON does not identify notebook_16."
    )

add_check(
    "completion_summary_reload",
    COMPLETION_PATH,
    "PASS",
    "Valid JSON",
    "Valid Notebook 16 completion summary",
)

registry = completion.get("artifacts", {})
if not isinstance(registry, dict):
    registry = {}

# ---------------------------------------------------------------------
# 3. Discover manifest paths using the runtime and completion registry
# ---------------------------------------------------------------------

def registry_path(*names):
    for name in names:
        item = registry.get(name)
        if isinstance(item, dict) and item.get("path"):
            return Path(item["path"])
        if isinstance(item, str) and item.strip():
            return Path(item)
    return None


fingerprint_candidate = (
    Path(FINGERPRINT_PATH)
    if "FINGERPRINT_PATH" in globals()
    else registry_path("fingerprint_manifest")
)

input_candidate = (
    Path(INPUT_MANIFEST_PATH)
    if "INPUT_MANIFEST_PATH" in globals()
    else registry_path("input_manifest")
)

validation_candidate = (
    Path(VALIDATION_PATH)
    if "VALIDATION_PATH" in globals()
    else registry_path("coverage_validation")
)

fingerprint_path = discover_manifest(
    [p for p in [fingerprint_candidate] if p is not None],
    [
        "*fingerprint*.json",
        "*fingerprint*.csv",
        "*hash_manifest*.json",
        "*hash_manifest*.csv",
    ],
)

input_manifest_path = discover_manifest(
    [p for p in [input_candidate] if p is not None],
    [
        "*input_manifest*.csv",
        "*input_manifest*.json",
        "*source_manifest*.csv",
        "*source_manifest*.json",
    ],
)

validation_path = discover_manifest(
    [p for p in [validation_candidate] if p is not None],
    ["privacy_coverage_validation.csv"],
)

for name, path in [
    ("fingerprint_manifest_discovery", fingerprint_path),
    ("input_manifest_discovery", input_manifest_path),
    ("coverage_validation_discovery", validation_path),
]:
    add_check(
        name,
        path if path else NOTEBOOK_ROOT,
        "PASS" if path is not None and path.is_file() else "FAIL",
        path if path else "NOT FOUND",
        "Existing manifest file",
    )

# ---------------------------------------------------------------------
# 4. Audit every artifact registered in completion JSON
# ---------------------------------------------------------------------

artifact_paths = {
    "completion_summary": COMPLETION_PATH,
}

for name, item in registry.items():
    raw_path = item.get("path") if isinstance(item, dict) else item
    if raw_path:
        artifact_paths[name] = resolve_path(raw_path)

if fingerprint_path:
    artifact_paths["fingerprint_manifest"] = fingerprint_path
if input_manifest_path:
    artifact_paths["input_manifest"] = input_manifest_path
if validation_path:
    artifact_paths["coverage_validation"] = validation_path

for name, path in artifact_paths.items():
    exists = path.is_file()
    size = path.stat().st_size if exists else 0

    add_check(
        "artifact_exists_and_nonempty",
        path,
        "PASS" if exists and size > 0 else "FAIL",
        f"exists={exists}; size_bytes={size}",
        "File exists and size > 0",
        name,
    )

    if not exists or size == 0:
        continue

    suffix = path.suffix.lower()

    if suffix == ".csv":
        try:
            df = read_csv(path)
            add_check(
                "csv_reload",
                path,
                "PASS",
                f"{len(df)} rows x {len(df.columns)} columns",
                "CSV can be reloaded",
                name,
            )
        except Exception as exc:
            add_check(
                "csv_reload", path, "FAIL",
                details=repr(exc),
            )

    elif suffix == ".json":
        try:
            read_json(path)
            add_check(
                "json_reload", path, "PASS",
                "JSON parsed", "Valid JSON", name,
            )
        except Exception as exc:
            add_check(
                "json_reload", path, "FAIL",
                details=repr(exc),
            )

# ---------------------------------------------------------------------
# 5. Flexible extraction of path/hash pairs
# ---------------------------------------------------------------------

PATH_KEYS = (
    "path", "file_path", "filepath", "file",
    "artifact_path", "source_path", "filename",
)
HASH_KEYS = (
    "sha256", "sha_256", "hash", "file_hash",
    "fingerprint", "sha256_hash",
)


def extract_pairs(obj):
    """
    Extract common manifest layouts:
      {"path": "...", "sha256": "..."}
      {"files": [{"path": "...", "sha256": "..."}]}
      {"path/to/file.csv": {"sha256": "..."}}
      {"path/to/file.csv": "64-character-hash"}
    """
    pairs = []

    def walk(value, inherited_path=None):
        if isinstance(value, list):
            for item in value:
                walk(item, inherited_path)
            return

        if not isinstance(value, dict):
            if inherited_path and isinstance(value, str) and valid_sha256(value):
                pairs.append((inherited_path, value))
            return

        lower = {str(k).strip().lower(): v for k, v in value.items()}

        path_value = next(
            (lower[k] for k in PATH_KEYS if k in lower),
            inherited_path,
        )
        hash_value = next(
            (lower[k] for k in HASH_KEYS if k in lower),
            None,
        )

        if path_value and hash_value:
            pairs.append((str(path_value), str(hash_value)))
            return

        for key, child in value.items():
            if isinstance(child, dict):
                walk(child, str(key))
            elif isinstance(child, list):
                walk(child, None)
            elif isinstance(child, str) and valid_sha256(child):
                pairs.append((str(key), child))

    walk(obj)

    # Remove exact duplicate path/hash pairs.
    unique = []
    seen = set()
    for pair in pairs:
        if pair not in seen:
            unique.append(pair)
            seen.add(pair)

    return unique


# ---------------------------------------------------------------------
# 6. Verify output fingerprint hashes
# ---------------------------------------------------------------------

fingerprint_pairs = []

if fingerprint_path and fingerprint_path.is_file():
    try:
        if fingerprint_path.suffix.lower() == ".csv":
            fingerprint_df = read_csv(fingerprint_path)
            fingerprint_pairs = extract_pairs(
                fingerprint_df.to_dict(orient="records")
            )
        else:
            fingerprint_pairs = extract_pairs(
                read_json(fingerprint_path)
            )

        add_check(
            "fingerprint_manifest_parsing",
            fingerprint_path,
            "PASS" if fingerprint_pairs else "FAIL",
            len(fingerprint_pairs),
            "At least one path/hash pair",
            "Manifest parsed using flexible schema extraction.",
        )

    except Exception as exc:
        add_check(
            "fingerprint_manifest_parsing",
            fingerprint_path,
            "FAIL",
            details=repr(exc),
        )

fingerprint_verified = 0
fingerprint_failed = 0

for raw_path, expected_hash in fingerprint_pairs:
    path = resolve_path(raw_path)

    if not valid_sha256(expected_hash):
        fingerprint_failed += 1
        add_check(
            "output_sha256",
            path,
            "FAIL",
            expected_hash,
            "Valid SHA-256",
            "Stored hash is malformed.",
        )
        continue

    if not path.is_file():
        fingerprint_failed += 1
        add_check(
            "output_sha256",
            path,
            "FAIL",
            "File missing",
            expected_hash,
        )
        continue

    actual_hash = sha256_file(path)
    matches = actual_hash.lower() == expected_hash.lower()

    if matches:
        fingerprint_verified += 1
    else:
        fingerprint_failed += 1

    add_check(
        "output_sha256",
        path,
        "PASS" if matches else "FAIL",
        actual_hash,
        expected_hash,
        "Hash matches" if matches else "Hash mismatch",
    )

# ---------------------------------------------------------------------
# 7. Verify source/input manifest hashes
# ---------------------------------------------------------------------

input_pairs = []

if input_manifest_path and input_manifest_path.is_file():
    try:
        if input_manifest_path.suffix.lower() == ".csv":
            input_df = read_csv(input_manifest_path)
            input_pairs = extract_pairs(
                input_df.to_dict(orient="records")
            )
        else:
            input_pairs = extract_pairs(
                read_json(input_manifest_path)
            )

        add_check(
            "input_manifest_parsing",
            input_manifest_path,
            "PASS" if input_pairs else "FAIL",
            len(input_pairs),
            "At least one path/hash pair",
        )

    except Exception as exc:
        add_check(
            "input_manifest_parsing",
            input_manifest_path,
            "FAIL",
            details=repr(exc),
        )

input_verified = 0
input_failed = 0

for raw_path, expected_hash in input_pairs:
    path = resolve_path(raw_path)

    if not valid_sha256(expected_hash):
        input_failed += 1
        add_check(
            "input_sha256",
            path,
            "FAIL",
            expected_hash,
            "Valid SHA-256",
            "Stored hash is malformed.",
        )
        continue

    if not path.is_file():
        input_failed += 1
        add_check(
            "input_sha256",
            path,
            "FAIL",
            "Source file missing",
            expected_hash,
        )
        continue

    actual_hash = sha256_file(path)
    matches = actual_hash.lower() == expected_hash.lower()

    if matches:
        input_verified += 1
    else:
        input_failed += 1

    add_check(
        "input_sha256",
        path,
        "PASS" if matches else "FAIL",
        actual_hash,
        expected_hash,
        "Hash matches" if matches else "Hash mismatch",
    )

# ---------------------------------------------------------------------
# 8. Validate coverage report and completion consistency
# ---------------------------------------------------------------------

validation_df = None
coverage_pass = False

if validation_path and validation_path.is_file():
    try:
        validation_df = read_csv(validation_path)

        required = {"check", "status"}
        valid_schema = required.issubset(validation_df.columns)

        coverage_pass = (
            valid_schema
            and not validation_df.empty
            and validation_df["status"].astype(str).eq("PASS").all()
        )

        add_check(
            "coverage_validation_contents",
            validation_path,
            "PASS" if coverage_pass else "FAIL",
            (
                validation_df["status"].astype(str).value_counts().to_dict()
                if valid_schema else list(validation_df.columns)
            ),
            "All checks PASS",
        )

    except Exception as exc:
        add_check(
            "coverage_validation_contents",
            validation_path,
            "FAIL",
            details=repr(exc),
        )

completion_validation_pass = (
    completion.get("validation_status") == "PASS"
    and completion.get("failed_validation_checks") == 0
)

add_check(
    "completion_validation_consistency",
    COMPLETION_PATH,
    "PASS" if completion_validation_pass and coverage_pass else "FAIL",
    {
        "completion_validation_status": completion.get("validation_status"),
        "completion_failed_checks": completion.get("failed_validation_checks"),
        "coverage_csv_all_pass": coverage_pass,
    },
    "Completion JSON and coverage report both PASS",
)

# ---------------------------------------------------------------------
# 9. Audit summary and persist results
# ---------------------------------------------------------------------

AUDIT_DF = pd.DataFrame(audit_rows)

if AUDIT_DF.empty:
    raise RuntimeError("No audit checks were generated.")

failed_checks = int(AUDIT_DF["status"].astype(str).ne("PASS").sum())
passed_checks = int(AUDIT_DF["status"].astype(str).eq("PASS").sum())

# Strict pass: require verifiable output hashes and input hashes.
hashes_pass = (
    len(fingerprint_pairs) > 0
    and fingerprint_verified == len(fingerprint_pairs)
    and fingerprint_failed == 0
    and len(input_pairs) > 0
    and input_verified == len(input_pairs)
    and input_failed == 0
)

audit_pass = failed_checks == 0 and hashes_pass

AUDIT_SUMMARY = {
    "audit_id": "notebook_16_final_artifact_audit",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "notebook_root": str(NOTEBOOK_ROOT),
    "audit_status": "PASS" if audit_pass else "FAIL",
    "passed_checks": passed_checks,
    "failed_checks": failed_checks,
    "registered_artifact_count": len(artifact_paths),
    "fingerprint_manifest_path": (
        str(fingerprint_path) if fingerprint_path else None
    ),
    "fingerprint_entries": len(fingerprint_pairs),
    "output_hashes_verified": fingerprint_verified,
    "output_hashes_failed": fingerprint_failed,
    "input_manifest_path": (
        str(input_manifest_path) if input_manifest_path else None
    ),
    "input_hash_entries": len(input_pairs),
    "input_hashes_verified": input_verified,
    "input_hashes_failed": input_failed,
    "coverage_validation_pass": bool(coverage_pass),
    "completion_validation_pass": bool(completion_validation_pass),
    "audit_csv_path": str(AUDIT_CSV_PATH),
    "audit_json_path": str(AUDIT_JSON_PATH),
    "note": (
        "A missing or unsupported manifest is reported as a failure. "
        "A hash audit is only complete when stored hashes can be parsed "
        "and compared with the current files."
    ),
}

AUDIT_DF.to_csv(AUDIT_CSV_PATH, index=False)

with open(AUDIT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(AUDIT_SUMMARY, f, indent=2, allow_nan=False)

# Reload audit outputs.
AUDIT_RELOADED = read_csv(AUDIT_CSV_PATH)
AUDIT_SUMMARY_RELOADED = read_json(AUDIT_JSON_PATH)

reload_pass = (
    len(AUDIT_RELOADED) == len(AUDIT_DF)
    and AUDIT_SUMMARY_RELOADED.get("audit_status")
        == AUDIT_SUMMARY.get("audit_status")
)

print("-" * 100)
print("Registered artifacts:", len(artifact_paths))
print("Fingerprint manifest:", fingerprint_path)
print("Fingerprint entries:", len(fingerprint_pairs))
print("Output hashes verified:", fingerprint_verified)
print("Output hashes failed:", fingerprint_failed)
print("Input manifest:", input_manifest_path)
print("Input hash entries:", len(input_pairs))
print("Input hashes verified:", input_verified)
print("Input hashes failed:", input_failed)
print("Coverage validation:", "PASS" if coverage_pass else "FAIL")
print("Passed checks:", passed_checks)
print("Failed checks:", failed_checks)
print("Audit CSV:", AUDIT_CSV_PATH)
print("Audit JSON:", AUDIT_JSON_PATH)
print("Audit output reload:", "PASS" if reload_pass else "FAIL")
print("FINAL AUDIT STATUS:", "PASS" if audit_pass and reload_pass else "FAIL")
print("=" * 100)

if not reload_pass:
    raise RuntimeError("Final audit output reload verification failed.")

if not audit_pass:
    print(
        "Inspect the audit CSV for exact failed checks. "
        "If a manifest has a different schema, adapt the parser to that "
        "actual schema; do not treat unverified hashes as verified."
    )
    raise RuntimeError(
        "Notebook 16 final artifact audit failed. "
        "The audit report has been saved for diagnosis."
    )

print("NOTEBOOK 16 ARTIFACT AUDIT: PASS")

SECTION 15 — FINAL ARTIFACT, HASH & REPRODUCIBILITY AUDIT
----------------------------------------------------------------------------------------------------
Registered artifacts: 12
Fingerprint manifest: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/inputs/real_data_fingerprints.csv
Fingerprint entries: 3
Output hashes verified: 3
Output hashes failed: 0
Input manifest: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/inputs/privacy_input_manifest.csv
Input hash entries: 6
Input hashes verified: 6
Input hashes failed: 0
Coverage validation: PASS
Passed checks: 41
Failed checks: 0
Audit CSV: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/audit/final_artifact_audit.csv
Audit JSON: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_16/audit/final_artifact_audit_summary.json
Audit output reload: PASS
FINAL AUDIT STATUS: PASS
NOTEBOOK 16 ARTIFACT AUDIT: PASS
